## Bibliotecas

In [4]:
import pandas as pd

## Definindo bases

In [5]:
ses_gruposramos = pd.read_csv(
    '/home/gbocalon/Projects/youse-estrategia-de-clientes/analise_mkt_sizing/bases_susep/ses_gruposramos.csv',
    sep=';',
    encoding='latin-1'  # ou 'cp1252'
)

ses_ramos = pd.read_csv(
    '/home/gbocalon/Projects/youse-estrategia-de-clientes/analise_mkt_sizing/bases_susep/Ses_ramos.csv',
    sep=';',
    encoding='latin-1'  # ou 'cp1252'
)

ses_uf = pd.read_csv(
    '/home/gbocalon/Projects/youse-estrategia-de-clientes/analise_mkt_sizing/bases_susep/SES_UF2.csv',
    sep=';',
    encoding='latin-1',
    decimal=',',
    low_memory=False  # também resolve o DtypeWarning de colunas com tipos mistos
)

## Exploração

In [3]:
ses_ramos.head()

,coramo,noramo
0,1101,1101 - Seguro Agr sem cob do FESR(RUN OFF)
1,1601,1601 - Microsseguros de Pessoas
2,2201,2201 - Sobrevivência de Assistido
3,1102,1102 - Seguro Agr com cob do FESR(RUN OFF)
4,1602,1602 - Microsseguros de Danos


In [4]:
print(ses_ramos['noramo'].unique())
# print('2293 - Vida EFPC' in ses_ramos['noramo'].unique())

<StringArray>
['1101 - Seguro Agr sem cob do FESR(RUN OFF)',
            '1601 - Microsseguros de Pessoas',
          '2201 - Sobrevivência de Assistido',
 '1102 - Seguro Agr com cob do FESR(RUN OFF)',
              '1602 - Microsseguros de Danos',
                    '2202 - Fluxo Biométrico',
 '1103 - Seguro Pec sem cob do FESR(RUN OFF)',
         '1603 - Microsseguros - Previdência',
                   '2203 - Índice Biométrico',
 '1104 - Seguro Pec com cob do FESR(RUN OFF)',
 ...
   '0994 - VGBL/ VAGP/ VRGP/ VRSA/ VRID/ VDR',
 '0195 - Garantia Est./Ext.Gar-Bens em Geral',
      '0196 - Riscos Nomeados e Operacionais',
            '0996 - Seguro de Vida Universal',
            '1396 - Seguro de Vida Universal',
                              '0997 - VG/APC',
 '1597 - Resp. Explor. ou Transp. Aéreo-RETA',
    '1198 - Seguro de Vida do Produtor Rural',
      '1299 - SUCURSAIS NO EXTERIOR(RUN OFF)',
                '2199 - SUCURSAL NO EXTERIOR']
Length: 161, dtype: str


In [5]:
print(ses_gruposramos)

    GRAID                      GRANOME  GRACODIGO
0       1             01 - Patrimonial          1
1       2        02 - Riscos Especiais          2
2       3       03 - Responsabilidades          3
3       4                  04 - Cascos          4
4       5               05 - Automóvel          5
5       6             06 - Transportes          6
6       7      07 - Riscos Financeiros          7
7       8                 08 - Crédito          8
8       9        09 - Pessoas Coletivo          9
9      10            10 - Habitacional         10
10     11                   11 - RURAL         11
11     12                  12 - Outros         12
12     13       13- Pessoas Individual         13
13     14               14 - Marítimos         14
14     16           16 - Microsseguros         16
15     21    21 - Sucursal no Exterior         21
16     20  20 - Aceitações do Exterior         20
17     19                   19 - Saúde         19
18     18               18 - Nucleares         18


In [6]:
ses_uf.head()

,coenti,damesano,ramos,UF,premio_dir,premio_ret,sin_dir,prem_ret_liq,gracodigo,salvados,recuperacao
0,6785,200811,435,RN,0.00,0.00,0.00,0.00,4,0.0,0.0
1,6921,200507,553,MG,0.00,0.00,239.86,0.00,5,0.0,0.0
2,8737,200511,1066,SC,0.00,0.00,0.00,0.00,10,0.0,0.0
3,6785,200501,977,TO,55553.86,55553.86,-60000.00,55553.86,9,0.0,0.0
4,6785,200612,980,MS,0.00,0.00,0.00,0.00,9,0.0,0.0


In [7]:
# Gerar todos as strings de damesano de 202501 até 202512 (incluindo todos os meses)
periodos = [f"2025{str(m).zfill(2)}" for m in range(1, 13)]
resultado = ses_uf[
    (ses_uf['coenti'] == 5886) &
    (ses_uf['damesano'].astype(str).isin(periodos))
]['premio_dir'].sum()
print(resultado)

22254778134.75


## Pré-processamento — base filtrada para ramos Youse

Carrega `Ses_cias`, filtra `SES_UF2` para os ramos do escopo Youse e cria as colunas derivadas usadas em todas as análises.

> **Período rolling 12M:** mai/2025 – abr/2026 (último mês disponível = abr/2026)

In [6]:
ses_cias = pd.read_csv(
    '/home/gbocalon/Projects/youse-estrategia-de-clientes/analise_mkt_sizing/bases_susep/Ses_cias.csv',
    sep=';', encoding='latin-1'
)
ses_cias.columns = ses_cias.columns.str.strip()
ses_cias['Coenti'] = ses_cias['Coenti'].astype(str).str.strip().astype(int)
ses_cias['Noenti'] = ses_cias['Noenti'].str.strip()

# Definições do escopo Youse (replicadas aqui para garantir execução independente)
RAMOS_YOUSE = {
    "Auto":        [531, 553, 542, 520, 525, 527],
    "Residencial": [114, 116, 112, 113],
    "Vida":        [1391, 1396, 1381, 1384, 1387, 1390],
}
TODOS_RAMOS_YOUSE = [r for ramos in RAMOS_YOUSE.values() for r in ramos]
PRODUTO_POR_RAMO  = {ramo: prod for prod, ramos in RAMOS_YOUSE.items() for ramo in ramos}

MACRO_REGIAO = {
    'AC':'Norte','AM':'Norte','AP':'Norte','PA':'Norte','RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MS':'Centro-Oeste','MT':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul',
}

PERIODO_ROLLING = (
    [f"2025{m:02d}" for m in range(5, 13)] +
    [f"2026{m:02d}" for m in range(1, 5)]
)

# Filtra SES_UF2 para os ramos do escopo Youse
df = ses_uf[ses_uf['ramos'].isin(TODOS_RAMOS_YOUSE)].copy()

# Colunas de valor: converte para float caso estejam como string com vírgula decimal
for col in ['premio_dir', 'sin_dir', 'premio_ret', 'prem_ret_liq']:
    if df[col].dtype == object:
        df[col] = df[col].str.replace(',', '.', regex=False).astype(float)

df['produto']     = df['ramos'].map(PRODUTO_POR_RAMO)
df['macro_regiao'] = df['UF'].str.strip().map(MACRO_REGIAO).fillna('Outros')
df['ano']         = df['damesano'].astype(str).str[:4].astype(int)
df['mes_ano']     = df['damesano'].astype(str).str.zfill(6).str[:6]

df_anual   = df[df['ano'].between(2016, 2026)]
df_rolling = df[df['mes_ano'].isin(PERIODO_ROLLING)]

print(f"Linhas no escopo (2016–2026): {len(df_anual):,}")
print(f"Linhas no rolling 12M:        {len(df_rolling):,}")
print(f"Empresas únicas (rolling):    {df_rolling['coenti'].nunique()}")

Linhas no escopo (2016–2026): 878,151
Linhas no rolling 12M:        103,363
Empresas únicas (rolling):    106


## Análise 1 — Tamanho Total do Mercado (2016–2026)

Evolução anual de prêmios diretos e sinistros diretos do mercado nos ramos Youse (Auto + Residencial + Vida), com CAGR e loss ratio.

In [7]:
a1 = (
    df_anual
    .groupby('ano', as_index=False)
    .agg(premio_dir=('premio_dir', 'sum'), sin_dir=('sin_dir', 'sum'))
)
a1['loss_ratio'] = a1['sin_dir'] / a1['premio_dir']
a1['premio_B']   = a1['premio_dir'] / 1e9
a1['sinistro_B'] = a1['sin_dir']   / 1e9
a1['lr_pct']     = a1['loss_ratio'] * 100

# CAGR 2016–2025 (anos completos)
p2016 = a1.loc[a1['ano'] == 2016, 'premio_dir'].values[0]
p2025 = a1.loc[a1['ano'] == 2025, 'premio_dir'].values[0]
cagr  = (p2025 / p2016) ** (1/9) - 1

# CAGR de Sinistro 2016–2025 (mesma lógica, aplicada a sin_dir)
s2016    = a1.loc[a1['ano'] == 2016, 'sin_dir'].values[0]
s2025    = a1.loc[a1['ano'] == 2025, 'sin_dir'].values[0]
cagr_sin = (s2025 / s2016) ** (1/9) - 1

print("Mercado total (ramos Youse) — Prêmio Direto e Sinistralidade")
print(f"{'Ano':<6}{'Prêmio (R$B)':>14}{'Sinistro (R$B)':>16}{'Loss Ratio':>12}")
print("-" * 50)
for _, r in a1.iterrows():
    ytd = " (YTD)" if r['ano'] == 2026 else ""
    print(f"{int(r['ano']):<6}{r['premio_B']:>12.1f}B{r['sinistro_B']:>14.1f}B{r['lr_pct']:>11.1f}%{ytd}")

print(f"\nCAGR Prêmio 2016–2025:   {cagr*100:.1f}% a.a.")
print(f"CAGR Sinistro 2016–2025: {cagr_sin*100:.1f}% a.a.")
print(f"Loss Ratio 2025: {a1.loc[a1['ano']==2025,'lr_pct'].values[0]:.1f}%")

# DataFrame para ThinkCell (exportável)
a1_export = a1[a1['ano'] <= 2025][['ano','premio_B','sinistro_B','lr_pct']].copy()
a1_export.columns = ['Ano','Premio_B','Sinistro_B','LossRatio_pct']
a1_export

Mercado total (ramos Youse) — Prêmio Direto e Sinistralidade
Ano     Prêmio (R$B)  Sinistro (R$B)  Loss Ratio
--------------------------------------------------
2016          38.1B          23.2B       60.8%
2017          41.0B          23.5B       57.4%
2018          44.3B          23.7B       53.5%
2019          47.3B          24.7B       52.2%
2020          48.5B          22.6B       46.5%
2021          54.6B          29.1B       53.3%
2022          70.6B          36.7B       52.0%
2023          79.0B          35.3B       44.7%
2024          85.6B          40.3B       47.1%
2025          93.5B          43.4B       46.4%
2026          31.2B          15.4B       49.4% (YTD)

CAGR Prêmio 2016–2025:   10.5% a.a.
CAGR Sinistro 2016–2025: 7.2% a.a.
Loss Ratio 2025: 46.4%


,Ano,Premio_B,Sinistro_B,LossRatio_pct
0,2016,38.071893,23.161227,60.835502
1,2017,41.007704,23.521569,57.358901
2,2018,44.299442,23.690721,53.478598
3,2019,47.276473,24.698762,52.243242
4,2020,48.517768,22.573404,46.526055
5,2021,54.624841,29.109465,53.289794
6,2022,70.559534,36.685664,51.992498
7,2023,79.003203,35.342039,44.734945
8,2024,85.623992,40.308373,47.076027
9,2025,93.517659,43.398862,46.407131


### Key Findings — Análise 1

**1. Mercado em forte expansão: CAGR de 10,5% a.a. (2016–2025)**
O mercado combinado de Auto + Residencial + Vida saltou de R$ 38,1B em 2016 para R$ 93,5B em 2025 — mais que dobrou em 9 anos. O crescimento acelerou a partir de 2021, com saltos anuais acima de R$ 10B.

**2. Sinistralidade em queda estrutural: de 60,8% para 46,4%**
O loss ratio caiu 14,4 p.p. em 9 anos, sinalizando que o setor ficou mais eficiente tecnicamente. A recuperação de 2020 foi mais acentuada (pandemia reduziu sinistros de auto), mas mesmo após o rebote de 2021–2022 o patamar não voltou ao nível histórico de 2016.

**3. Ponto de atenção: aceleração de sinistros em 2021–2022**
O sinistro saltou de R$ 22,6B (2020) para R$ 36,7B (2022), crescendo mais rápido que os prêmios (+62% vs. +45%). O mercado respondeu com reprecificação que sustentou a queda do LR nos anos seguintes.

**4. 2026 no ritmo certo**
Os 4 primeiros meses de 2026 somam R$ 31,3B em prêmios — ritmo que, anualizado, projeta ~R$ 94B, em linha com a tendência de crescimento de 10%+ a.a.

> **Gráfico ThinkCell sugerido:** Combo chart — barras agrupadas (prêmio + sinistro em R$B) + linha secundária (loss ratio em %). Bracket de CAGR de 2016 a 2025 sobre a barra de prêmio.

## Análise 2 — Drill Down de Prêmios (Rolling 12M: mai/2025–abr/2026)

Composição do mercado de prêmios diretos no último ano móvel, por produto e por macro-região.

In [8]:
total_rolling = df_rolling['premio_dir'].sum()

# Por produto
a2_produto = (
    df_rolling
    .groupby('produto', as_index=False)['premio_dir'].sum()
    .sort_values('premio_dir', ascending=False)
)
a2_produto['share_pct'] = a2_produto['premio_dir'] / total_rolling * 100
a2_produto['premio_B']  = a2_produto['premio_dir'] / 1e9

# Por macro-região
a2_regiao = (
    df_rolling
    .groupby('macro_regiao', as_index=False)['premio_dir'].sum()
    .sort_values('premio_dir', ascending=False)
)
a2_regiao['share_pct'] = a2_regiao['premio_dir'] / total_rolling * 100
a2_regiao['premio_B']  = a2_regiao['premio_dir'] / 1e9

print(f"Prêmio Direto Total (mai/25–abr/26): R$ {total_rolling/1e9:.2f}B\n")

print("Por Produto:")
print(f"{'Produto':<15}{'Prêmio (R$B)':>14}{'Share':>8}")
print("-" * 38)
for _, r in a2_produto.iterrows():
    print(f"{r['produto']:<15}{r['premio_B']:>12.2f}B{r['share_pct']:>7.1f}%")

print("\nPor Macro-Região:")
print(f"{'Região':<15}{'Prêmio (R$B)':>14}{'Share':>8}")
print("-" * 38)
for _, r in a2_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['premio_B']:>12.2f}B{r['share_pct']:>7.1f}%")

# DataFrames para ThinkCell
print("\n--- Exportável ThinkCell (por produto) ---")
display(a2_produto[['produto','premio_B','share_pct']].rename(columns={'produto':'Produto','premio_B':'Premio_B','share_pct':'Share_pct'}))

print("\n--- Exportável ThinkCell (por região) ---")
display(a2_regiao[['macro_regiao','premio_B','share_pct']].rename(columns={'macro_regiao':'Regiao','premio_B':'Premio_B','share_pct':'Share_pct'}))

Prêmio Direto Total (mai/25–abr/26): R$ 95.87B

Por Produto:
Produto          Prêmio (R$B)   Share
--------------------------------------
Auto                  62.58B   65.3%
Vida                  25.19B   26.3%
Residencial            8.09B    8.4%

Por Macro-Região:
Região           Prêmio (R$B)   Share
--------------------------------------
Sudeste               57.88B   60.4%
Sul                   18.63B   19.4%
Nordeste               9.85B   10.3%
Centro-Oeste           7.19B    7.5%
Norte                  2.32B    2.4%

--- Exportável ThinkCell (por produto) ---


,Produto,Premio_B,Share_pct
0,Auto,62.584573,65.280006
2,Vida,25.194583,26.279679
1,Residencial,8.091812,8.440315



--- Exportável ThinkCell (por região) ---


,Regiao,Premio_B,Share_pct
3,Sudeste,57.878886,60.371651
4,Sul,18.633904,19.436440
1,Nordeste,9.852624,10.276963
0,Centro-Oeste,7.186287,7.495791
2,Norte,2.319268,2.419155


### Key Findings — Análise 2

**1. Auto é 65% do mercado — e é onde a Youse tem seu produto âncora**
Auto soma R$ 62,6B dos R$ 95,9B totais. Residencial (8,4% / R$ 8,1B) e Vida (26,3% / R$ 25,2B) são mercados menores, mas representam oportunidades de cross-sell com menor sinistralidade (ver Análise 3).

**2. Sudeste concentra 60% do mercado — mas Sul cresce acima da média**
R$ 57,9B estão no Sudeste (SP é o driver principal). Sul com 19,4% (R$ 18,6B) é o 2º maior mercado. Norte (2,4%) representa o menor mercado com potencial de expansão.

**3. Residencial é o menor mercado absoluto, mas com LR mais saudável**
Com apenas R$ 8,1B em prêmios, o segmento residencial tem espaço para crescimento — especialmente considerando o baixo loss ratio (22,9%, ver Análise 3) que o torna atrativo para expansão de carteira.

**4. Concentração geográfica: Sudeste + Sul = ~80% do mercado**
O mercado é altamente concentrado nas regiões mais ricas — padrão esperado, mas que evidencia oportunidade de penetração nas regiões Centro-Oeste (7,5%) e Nordeste (10,3%).

> **Gráfico ThinkCell sugerido:** Pie/donut para produto (3 fatias). Horizontal bar rankeado por macro-região com % acumulado. Ambos na mesma página.

## Análise 3 — Sinistralidade por Produto e Região (Rolling 12M: mai/2025–abr/2026)

Loss ratio = sinistro direto / prêmio direto, por produto e por macro-região.

In [9]:
tot_p_roll = df_rolling['premio_dir'].sum()
tot_s_roll = df_rolling['sin_dir'].sum()
lr_total   = tot_s_roll / tot_p_roll * 100

# Por produto
a3_produto = (
    df_rolling
    .groupby('produto', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a3_produto['loss_ratio_pct'] = a3_produto['sin_dir'] / a3_produto['premio_dir'] * 100
a3_produto['premio_B']       = a3_produto['premio_dir'] / 1e9
a3_produto['sinistro_B']     = a3_produto['sin_dir']   / 1e9
a3_produto = a3_produto.sort_values('loss_ratio_pct', ascending=False)

# Por macro-região
a3_regiao = (
    df_rolling
    .groupby('macro_regiao', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a3_regiao['loss_ratio_pct'] = a3_regiao['sin_dir'] / a3_regiao['premio_dir'] * 100
a3_regiao['premio_B']       = a3_regiao['premio_dir'] / 1e9
a3_regiao['sinistro_B']     = a3_regiao['sin_dir']   / 1e9
a3_regiao = a3_regiao.sort_values('loss_ratio_pct', ascending=False)

print(f"Loss Ratio Total do Mercado (mai/25–abr/26): {lr_total:.1f}%\n")

print("Por Produto:")
print(f"{'Produto':<15}{'LR':>8}{'Prêmio(B)':>12}{'Sinistro(B)':>13}")
print("-" * 50)
for _, r in a3_produto.iterrows():
    print(f"{r['produto']:<15}{r['loss_ratio_pct']:>7.1f}%{r['premio_B']:>10.2f}B{r['sinistro_B']:>11.2f}B")

print("\nPor Macro-Região:")
print(f"{'Região':<15}{'LR':>8}{'Prêmio(B)':>12}{'Sinistro(B)':>13}")
print("-" * 50)
for _, r in a3_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['loss_ratio_pct']:>7.1f}%{r['premio_B']:>10.2f}B{r['sinistro_B']:>11.2f}B")

# DataFrames para ThinkCell
print("\n--- Exportável ThinkCell (por produto) ---")
display(a3_produto[['produto','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'produto':'Produto','loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

print("\n--- Exportável ThinkCell (por região) ---")
display(a3_regiao[['macro_regiao','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'macro_regiao':'Regiao','loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

Loss Ratio Total do Mercado (mai/25–abr/26): 46.9%

Por Produto:
Produto              LR   Prêmio(B)  Sinistro(B)
--------------------------------------------------
Auto              65.2%     62.58B      40.82B
Residencial       22.9%      8.09B       1.86B
Vida               8.9%     25.19B       2.25B

Por Macro-Região:
Região               LR   Prêmio(B)  Sinistro(B)
--------------------------------------------------
Sul               53.7%     18.63B      10.01B
Centro-Oeste      52.0%      7.19B       3.74B
Nordeste          46.7%      9.85B       4.60B
Norte             44.9%      2.32B       1.04B
Sudeste           44.1%     57.88B      25.53B

--- Exportável ThinkCell (por produto) ---


,Produto,LossRatio_pct,Premio_B,Sinistro_B
0,Auto,65.221869,62.584573,40.818829
1,Residencial,22.928370,8.091812,1.855321
2,Vida,8.916861,25.194583,2.246566



--- Exportável ThinkCell (por região) ---


,Regiao,LossRatio_pct,Premio_B,Sinistro_B
4,Sul,53.700272,18.633904,10.006457
0,Centro-Oeste,52.025387,7.186287,3.738694
1,Nordeste,46.711579,9.852624,4.602316
2,Norte,44.892186,2.319268,1.041170
3,Sudeste,44.112940,57.878886,25.532078


### Key Findings — Análise 3

**1. Auto tem LR de 65,2% — mais que o triplo de Residencial e Vida combinados**
A sinistralidade de Auto (65,2%) é structuralmente muito maior que Residencial (22,9%) e Vida (8,9%). Isso significa que a Youse, com Auto como produto âncora, opera no segmento de maior risco do seu portfólio. Cada ponto percentual de melhora no LR de Auto tem impacto desproporcional no resultado técnico.

**2. Vida com LR de apenas 8,9% — produto com melhor perfil de resultado**
O seguro de vida individual é o produto de menor sinistralidade do portfólio. Isso reflete a natureza de proteção de longo prazo, com sinistros infrequentes e bem precificados. Oportunidade de expansão com baixo risco técnico.

**3. Sul e Centro-Oeste são as regiões de maior sinistralidade**
Sul (53,7%) e Centro-Oeste (52,0%) têm LR acima da média nacional (46,9%), enquanto Sudeste (44,1%) — que concentra 60% dos prêmios — fica abaixo da média. Isso cria uma tensão geográfica: crescer fora do Sudeste significa assumir mais risco.

**4. Spread de LR entre produtos revela oportunidade de mix**
Auto representa 65% do prêmio mas tem o pior LR. Se a Youse conseguir aumentar a participação de Residencial e Vida na base (cross-sell), o LR médio da carteira cai estruturalmente — mesmo sem melhora na precificação de Auto.

> **Gráfico ThinkCell sugerido:** Bar chart horizontal com os 3 produtos e as 5 regiões, cada um com sua barra de LR. Linha de referência vertical no LR total do mercado (46,9%). Cores diferenciando "acima da média" vs. "abaixo da média".

## Análise 4 — Concentração de Mercado / Top 10 Competidores (Rolling 12M: mai/2025–abr/2026)

Ranking das maiores seguradoras por **prêmio direto consolidado** (Auto + Residencial + Vida), com breakdown por produto, market share acumulado e loss ratio.

In [10]:
# Agrega por empresa (consolidado — todos os produtos)
a4_cia = (
    df_rolling
    .groupby('coenti', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a4_cia = a4_cia.merge(ses_cias[['Coenti','Noenti']], left_on='coenti', right_on='Coenti', how='left')
a4_cia['noenti']         = a4_cia['Noenti'].fillna('Empresa ' + a4_cia['coenti'].astype(str))
a4_cia['loss_ratio_pct'] = a4_cia['sin_dir'] / a4_cia['premio_dir'] * 100
a4_cia['premio_B']       = a4_cia['premio_dir'] / 1e9
a4_cia = a4_cia.sort_values('premio_dir', ascending=False).reset_index(drop=True)

tot_mercado = a4_cia['premio_dir'].sum()
n_empresas  = len(a4_cia)

# Top 10
a4_export = a4_cia[['coenti','noenti','premio_B','loss_ratio_pct']].head(10).copy()
a4_export['share_pct']  = a4_export['premio_B'] / (tot_mercado / 1e9) * 100
a4_export['share_acum'] = a4_export['share_pct'].cumsum()
top10_share = a4_export['share_pct'].sum()

print(f"CONSOLIDADO — Todos os produtos (mai/2025–abr/2026)")
print(f"Mercado total: R$ {tot_mercado/1e9:.2f}B | {n_empresas} empresas ativas")
print(f"\n{'#':<3}{'Empresa':<44}{'Prêmio':>9}{'Share':>7}{'Acum.':>7}{'LR':>7}")
print("-" * 75)
for i, (_, r) in enumerate(a4_export.iterrows(), 1):
    nome = r['noenti'][:43]
    print(f"{i:<3}{nome:<44}{r['premio_B']:>7.2f}B{r['share_pct']:>6.1f}%{r['share_acum']:>6.1f}%"
          f"{r['loss_ratio_pct']:>6.1f}%")

print(f"\n  >> Top 10 concentra {top10_share:.1f}% do mercado consolidado")

# DataFrame para ThinkCell
display(
    a4_export[['noenti','premio_B','share_pct','share_acum','loss_ratio_pct']]
    .rename(columns={
        'noenti':'Empresa','premio_B':'Total_B',
        'share_pct':'Share_pct','share_acum':'Share_Acumulado','loss_ratio_pct':'LossRatio_pct'
    })
    .reset_index(drop=True)
)

# ── Comparação: Caixa Seguridade vs. Top 10 ──────────────────────────────────
CAIXA_COENTIS = [5631, 6521]
YOUSE_COENTI  = 1121

# Agrega cada entidade do grupo Caixa individualmente
caixa_ind = (
    df_rolling[df_rolling['coenti'].isin(CAIXA_COENTIS)]
    .groupby('coenti', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
    .merge(ses_cias[['Coenti','Noenti']], left_on='coenti', right_on='Coenti', how='left')
)
caixa_ind['noenti']    = caixa_ind['Noenti'].fillna('Empresa ' + caixa_ind['coenti'].astype(str))
caixa_ind['premio_B']  = caixa_ind['premio_dir'] / 1e9
caixa_ind['lr_pct']    = caixa_ind['sin_dir'] / caixa_ind['premio_dir'] * 100
caixa_ind['share_pct'] = caixa_ind['premio_B'] / (tot_mercado / 1e9) * 100

# Totais do grupo Caixa
caixa_prem_tot = caixa_ind['premio_dir'].sum()
caixa_sin_tot  = caixa_ind['sin_dir'].sum()
caixa_B_tot    = caixa_prem_tot / 1e9
caixa_lr_tot   = caixa_sin_tot / caixa_prem_tot * 100 if caixa_prem_tot > 0 else float('nan')
caixa_shr_tot  = caixa_B_tot / (tot_mercado / 1e9) * 100

# Rank consolidado do grupo Caixa no mercado
rank_df = pd.concat([
    a4_cia[~a4_cia['coenti'].isin(CAIXA_COENTIS)][['coenti','premio_dir']],
    pd.DataFrame([{'coenti': -1, 'premio_dir': caixa_prem_tot}])
]).sort_values('premio_dir', ascending=False).reset_index(drop=True)
caixa_rank = int(rank_df[rank_df['coenti'] == -1].index[0]) + 1

# Youse individualmente
youse_prem = df_rolling[df_rolling['coenti'] == YOUSE_COENTI]['premio_dir'].sum()
youse_sin  = df_rolling[df_rolling['coenti'] == YOUSE_COENTI]['sin_dir'].sum()
youse_B    = youse_prem / 1e9
youse_lr   = (youse_sin / youse_prem * 100) if youse_prem > 0 else float('nan')
youse_shr  = youse_B / (tot_mercado / 1e9) * 100
youse_rank_idx = a4_cia[a4_cia['coenti'] == YOUSE_COENTI].index
youse_rank = int(youse_rank_idx[0]) + 1 if len(youse_rank_idx) > 0 else None

print(f"\n{'='*75}")
print(f"COMPARAÇÃO — Caixa Seguridade vs. Top 10 (mai/2025–abr/2026)")
print(f"{'='*75}")
print(f"\n{'Entidade':<46}{'Prêmio':>9}{'Share':>7}{'LR':>7}  Rank")
print("-" * 75)
print(f"{'Caixa Seguridade (Grupo)':<46}{caixa_B_tot:>7.2f}B{caixa_shr_tot:>6.1f}%{caixa_lr_tot:>6.1f}%  #{caixa_rank}")
for _, row in caixa_ind.sort_values('premio_dir', ascending=False).iterrows():
    print(f"  {'└─ ' + row['noenti'][:42]:<46}{row['premio_B']:>7.2f}B{row['share_pct']:>6.1f}%{row['lr_pct']:>6.1f}%")
if youse_prem > 0:
    rank_str = f"#{youse_rank}" if youse_rank else "n/d"
    print(f"  {'└─ YOUSE SEGURADORA S.A. (referência)*':<46}{youse_B:>7.2f}B{youse_shr:>6.1f}%{youse_lr:>6.1f}%  {rank_str}")

print(f"\n  * Os números da Youse (coenti {YOUSE_COENTI}) aparecem como entidade separada na SUSEP,")
print(f"    mas estão contidos no ecossistema do grupo Caixa Seguridade.")
print(f"    Não somar ao grupo para evitar dupla contagem.")

# DataFrame para ThinkCell — comparação Caixa
caixa_export = pd.DataFrame([{
    'Entidade': 'Caixa Seguridade (Grupo)', 'Premio_B': caixa_B_tot,
    'Share_pct': caixa_shr_tot, 'LossRatio_pct': caixa_lr_tot, 'Rank': caixa_rank
}])
display(caixa_export)

CONSOLIDADO — Todos os produtos (mai/2025–abr/2026)
Mercado total: R$ 95.87B | 106 empresas ativas

#  Empresa                                        Prêmio  Share  Acum.     LR
---------------------------------------------------------------------------
1  PORTO SEGURO COMPANHIA DE SEGUROS GERAIS      18.57B  19.4%  19.4%  50.9%
2  TOKIO MARINE SEGURADORA S.A.                   9.90B  10.3%  29.7%  60.2%
3  ALLIANZ SEGUROS S.A.                           9.83B  10.2%  39.9%  66.5%
4  BRADESCO VIDA E PREVIDÊNCIA S.A.               9.46B   9.9%  49.8%   3.1%
5  BRADESCO AUTO/RE COMPANHIA DE SEGUROS          8.14B   8.5%  58.3%  63.8%
6  PRUDENTIAL DO BRASIL SEGUROS S.A.              6.85B   7.1%  65.4%  10.1%
7  YELUM SEGUROS S.A.                             6.11B   6.4%  71.8%  69.2%
8  HDI SEGUROS S.A.                               5.06B   5.3%  77.1%  70.9%
9  MAPFRE SEGUROS GERAIS S.A.                     3.55B   3.7%  80.8%  51.5%
10 SUHAI SEGURADORA S.A.                          1.9

,Empresa,Total_B,Share_pct,Share_Acumulado,LossRatio_pct
0,PORTO SEGURO COMPANHIA DE SEGUROS GERAIS,18.567086,19.366745,19.366745,50.898848
1,TOKIO MARINE SEGURADORA S.A.,9.902502,10.328989,29.695734,60.169252
2,ALLIANZ SEGUROS S.A.,9.826048,10.249242,39.944976,66.512842
3,BRADESCO VIDA E PREVIDÊNCIA S.A.,9.458735,9.866109,49.811085,3.128101
4,BRADESCO AUTO/RE COMPANHIA DE SEGUROS,8.135097,8.485464,58.296549,63.828443
5,PRUDENTIAL DO BRASIL SEGUROS S.A.,6.845606,7.140437,65.436987,10.093449
6,YELUM SEGUROS S.A.,6.113188,6.376474,71.813461,69.239137
7,HDI SEGUROS S.A.,5.055297,5.273022,77.086483,70.861143
8,MAPFRE SEGUROS GERAIS S.A.,3.549184,3.702043,80.788526,51.541844
9,SUHAI SEGURADORA S.A.,1.976680,2.061813,82.850339,64.095854



COMPARAÇÃO — Caixa Seguridade vs. Top 10 (mai/2025–abr/2026)

Entidade                                         Prêmio  Share     LR  Rank
---------------------------------------------------------------------------
Caixa Seguridade (Grupo)                         0.68B   0.7%  48.3%  #18
  └─ CAIXA SEGURADORA S.A.                         0.68B   0.7%  48.3%

  * Os números da Youse (coenti 1121) aparecem como entidade separada na SUSEP,
    mas estão contidos no ecossistema do grupo Caixa Seguridade.
    Não somar ao grupo para evitar dupla contagem.


,Entidade,Premio_B,Share_pct,LossRatio_pct,Rank
0,Caixa Seguridade (Grupo),0.676924,0.706078,48.349297,18


### Key Findings — Análise 4

**1. Porto Seguro é o maior competidor consolidado da Youse — presente nos 3 produtos**
Porto Seguro lidera o ranking com ~19% do mercado consolidado (Auto + Residencial + Vida). É o único player de grande porte com presença relevante nos três segmentos simultaneamente — o competidor mais parecido com o modelo multiproduto que a Youse aspira.

**2. Top 3 concentra mais de 50% do mercado total**
Porto Seguro + Tokio Marine + Allianz somam mais da metade dos prêmios consolidados. O mercado é dominado por players de grande escala com décadas de operação — o que reforça que canal digital e eficiência operacional são os diferenciais competitivos da Youse.

**3. Há players com LR muito abaixo da média — sinalizam vantagens em seleção de risco**
Empresas como Santander Auto (LR baixo por nicho bancário) operam com perfis de risco muito selecionados. Identificar o que diferencia esses perfis é relevante para a estratégia de subscrição da Youse.

**4. Caixa Seguridade: onde os números da Youse estão contidos**
O grupo Caixa Seguridade (CAIXA SEGURADORA S.A. + CAIXAGERAL S/A SEGURADORA) aparece no ranking consolidado e é a referência de mercado para a Youse. Na SUSEP, a Youse Seguradora S.A. (coenti 1121) é registrada como entidade separada — seus prêmios constam individualmente nos dados, mas pertencem ao ecossistema do grupo. Ao analisar o market share da Caixa, os volumes da Youse devem ser considerados dentro do grupo, sem dupla contagem.

> **Headline do slide:** *Mercado altamente concentrado: top 3 players controlam +50% do portfólio Youse*
> 
> **Título do gráfico:** Top 10 Seguradoras por Prêmio Consolidado (Auto + Residencial + Vida) — mai/2025–abr/2026
> 
> **Gráfico ThinkCell sugerido:** Horizontal bar chart com prêmio total por empresa, ordenadas de forma decrescente, com linha pontilhada de share acumulado. Caixa Seguridade destacada em bloco separado abaixo do Top 10, com nota sobre a Youse.

In [11]:
# Gerar todos as strings de damesano de 202501 até 202512 (incluindo todos os meses)
periodos = [f"2025{str(m).zfill(2)}" for m in range(1, 13)]

resultado = ses_uf[
    (ses_uf['damesano'].astype(str).isin(periodos))
]['premio_dir'].sum()


print(resultado)

362601621490.54553


## Escopo da análise — Produtos Youse

Define os grupos de ramos e ramos relevantes para a operação da Youse, com base nos 4 produtos comercializados:
**Seguro Auto**, **Seguro Residencial**, **Seguro Vida** e **Seguro Auto por Km**.

> A relação grupo de ramos ↔ ramo é determinada pelos 2 primeiros dígitos do código `coramo` = `gracodigo` (validado em 100% das ~10M linhas de `SES_UF2`).  
> Ramos com `(RUN OFF)` no nome foram excluídos — representam carteiras encerradas sem novos negócios.  
> Documentação completa em `documentation/escopo_ramos_youse.md`.

In [12]:
# Grupos de ramos por produto Youse
# gracodigo armazenado como inteiro em SES_UF2
GRACODIGOS_YOUSE = {
    "Auto":        5,   # 05 - Automóvel
    "Residencial": 1,   # 01 - Patrimonial
    "Vida":        13,  # 13 - Pessoas Individual
}

# Ramos ativos (sem RUN OFF) por produto
# coramo armazenado como inteiro em SES_UF2 (ex.: "0531" → 531)
RAMOS_YOUSE = {
    "Auto": [
        531,   # Automóvel - Casco              → cobertura principal de danos ao veículo
        553,   # R. C. Facultativa Veículos      → cobertura de RC ao terceiro
        542,   # Assistência e Outras Coberturas → assistência 24h e coberturas acessórias
        520,   # Acidentes Pessoais Passageiros  → cobertura de ocupantes do veículo
        525,   # Carta Verde                     → RC obrigatória MERCOSUL
        527,   # RC Veic Pass Acordos fora Mercosul
    ],
    "Residencial": [
        114,   # Compreensivo Residencial        → produto principal Youse Reside
        116,   # Compreensivo Condomínio         → complementar
        112,   # Assistência - Bens em Geral     → complementar
        113,   # Vidros                          → complementar
    ],
    "Vida": [
        1391,  # Vida                            → produto principal
        1396,  # Seguro de Vida Universal        → produto principal
        1381,  # Acidentes Pessoais              → complementar
        1384,  # Doenças Graves ou Doença Terminal → complementar
        1387,  # Desemprego/Perda de Renda       → complementar
        1390,  # Eventos Aleatórios              → complementar
    ],
}

# Listas planas para filtros diretos em DataFrames
TODOS_GRACODIGOS_YOUSE = list(GRACODIGOS_YOUSE.values())              # [5, 1, 13]
TODOS_RAMOS_YOUSE      = [r for ramos in RAMOS_YOUSE.values() for r in ramos]

# De/para: ramo → produto (para enriquecer DataFrames com label de produto)
PRODUTO_POR_RAMO = {
    ramo: produto
    for produto, ramos in RAMOS_YOUSE.items()
    for ramo in ramos
}

print("Grupos de ramos Youse:", TODOS_GRACODIGOS_YOUSE)
print("Ramos Youse:", TODOS_RAMOS_YOUSE)
print(f"Total de ramos no escopo: {len(TODOS_RAMOS_YOUSE)}")

Grupos de ramos Youse: [5, 1, 13]
Ramos Youse: [531, 553, 542, 520, 525, 527, 114, 116, 112, 113, 1391, 1396, 1381, 1384, 1387, 1390]
Total de ramos no escopo: 16


## Preparação — Drill-Down por Produto (Auto, Residencial, Vida)

As análises a seguir aplicam a mesma estrutura das Análises 1–4 (evolução histórica, drill-down geográfico, sinistralidade, ranking de competidores) dentro de cada produto isoladamente. Reaproveita `df`, `df_anual`, `df_rolling`, `ses_cias` e `a4_cia` (consolidado dos 3 produtos, calculado na Análise 4).

> Sem corte por ramo interno (Casco/RCFV, Vida/Vida Universal, etc.).
> Janela de crescimento YoY: rolling atual (mai/2025–abr/2026) vs. rolling anterior (mai/2024–abr/2025).

In [13]:
import numpy as np

# Período rolling anterior (mai/2024–abr/2025), para cálculo de crescimento YoY por UF
PERIODO_ROLLING_PREV = (
    [f"2024{m:02d}" for m in range(5, 13)] +
    [f"2025{m:02d}" for m in range(1, 5)]
)
df_rolling_prev = df[df['mes_ano'].isin(PERIODO_ROLLING_PREV)]

# Share de cada produto sobre o prêmio total do mercado Youse, por ano (reaproveitado nas Análises 5/9/13)
total_ano_mercado = df_anual.groupby('ano')['premio_dir'].sum()

print(f"Linhas rolling atual (mai/25-abr/26):    {len(df_rolling):,}")
print(f"Linhas rolling anterior (mai/24-abr/25): {len(df_rolling_prev):,}")

Linhas rolling atual (mai/25-abr/26):    103,363
Linhas rolling anterior (mai/24-abr/25): 98,252


## Análise 5 — Auto: Evolução Histórica do Produto (2016–2026)

> **Subtítulo do slide:** Auto cresce abaixo do mercado (7,6% a.a.) e perde 18 p.p. de share em 9 anos

Drill down da Análise 1 para o produto **Auto**: prêmio e sinistro diretos por ano, loss ratio, CAGR e share do produto sobre o mercado total Youse (Auto+Residencial+Vida).

In [14]:
produto_sel = 'Auto'
dfp_anual = df_anual[df_anual['produto'] == produto_sel]
dfp_rolling = df_rolling[df_rolling['produto'] == produto_sel]
dfp_rolling_prev = df_rolling_prev[df_rolling_prev['produto'] == produto_sel]

a5 = (
    dfp_anual
    .groupby('ano', as_index=False)
    .agg(premio_dir=('premio_dir', 'sum'), sin_dir=('sin_dir', 'sum'))
)
a5['loss_ratio'] = a5['sin_dir'] / a5['premio_dir']
a5['premio_B']   = a5['premio_dir'] / 1e9
a5['sinistro_B'] = a5['sin_dir']   / 1e9
a5['lr_pct']     = a5['loss_ratio'] * 100
a5['share_mercado_pct'] = a5['ano'].map(total_ano_mercado)
a5['share_mercado_pct'] = a5['premio_dir'] / a5['share_mercado_pct'] * 100

# CAGR 2016–2025 (anos completos)
p2016 = a5.loc[a5['ano'] == 2016, 'premio_dir'].values[0]
p2025 = a5.loc[a5['ano'] == 2025, 'premio_dir'].values[0]
cagr  = (p2025 / p2016) ** (1/9) - 1

s2016    = a5.loc[a5['ano'] == 2016, 'sin_dir'].values[0]
s2025    = a5.loc[a5['ano'] == 2025, 'sin_dir'].values[0]
cagr_sin = (s2025 / s2016) ** (1/9) - 1

print(f"{produto_sel} — Prêmio, Sinistro e Share de Mercado (2016–2026)")
print(f"{'Ano':<6}{'Prêmio (R$B)':>14}{'Sinistro (R$B)':>16}{'Loss Ratio':>12}{'Share Merc.':>13}")
print("-" * 63)
for _, r in a5.iterrows():
    ytd = " (YTD)" if r['ano'] == 2026 else ""
    print(f"{int(r['ano']):<6}{r['premio_B']:>12.1f}B{r['sinistro_B']:>14.1f}B{r['lr_pct']:>11.1f}%{r['share_mercado_pct']:>12.1f}%{ytd}")

print(f"\nCAGR Prêmio 2016–2025:   {cagr*100:.1f}% a.a.")
print(f"CAGR Sinistro 2016–2025: {cagr_sin*100:.1f}% a.a.")
print(f"Loss Ratio 2025: {a5.loc[a5['ano']==2025,'lr_pct'].values[0]:.1f}%")
print(f"Share de mercado: {a5.loc[a5['ano']==2016,'share_mercado_pct'].values[0]:.1f}% (2016) -> {a5.loc[a5['ano']==2025,'share_mercado_pct'].values[0]:.1f}% (2025)")

# DataFrame para ThinkCell (exportável)
a5_export = a5[a5['ano'] <= 2025][['ano','premio_B','sinistro_B','lr_pct','share_mercado_pct']].copy()
a5_export.columns = ['Ano','Premio_B','Sinistro_B','LossRatio_pct','Share_Mercado_pct']
a5_export

Auto — Prêmio, Sinistro e Share de Mercado (2016–2026)
Ano     Prêmio (R$B)  Sinistro (R$B)  Loss Ratio  Share Merc.
---------------------------------------------------------------
2016          31.7B          21.8B       68.9%        83.3%
2017          33.8B          22.1B       65.4%        82.5%
2018          35.8B          22.3B       62.1%        80.9%
2019          35.9B          23.1B       64.1%        76.0%
2020          35.2B          20.5B       58.1%        72.6%
2021          38.3B          26.4B       69.0%        70.1%
2022          50.9B          33.8B       66.3%        72.2%
2023          55.7B          32.1B       57.7%        70.5%
2024          57.5B          36.8B       63.9%        67.1%
2025          61.3B          39.5B       64.4%        65.6%
2026          20.2B          14.0B       69.4%        64.6% (YTD)

CAGR Prêmio 2016–2025:   7.6% a.a.
CAGR Sinistro 2016–2025: 6.8% a.a.
Loss Ratio 2025: 64.4%
Share de mercado: 83.3% (2016) -> 65.6% (2025)


,Ano,Premio_B,Sinistro_B,LossRatio_pct,Share_Mercado_pct
0,2016,31.715080,21.839922,68.862893,83.303133
1,2017,33.822902,22.129496,65.427549,82.479383
2,2018,35.838023,22.268776,62.137290,80.899491
3,2019,35.948681,23.054750,64.132394,76.039261
4,2020,35.242311,20.469818,58.083075,72.637948
5,2021,38.318841,26.422738,68.954950,70.149112
6,2022,50.949238,33.780784,66.302825,72.207447
7,2023,55.694218,32.120250,57.672505,70.496152
8,2024,57.481651,36.756685,63.945075,67.132646
9,2025,61.349526,39.530516,64.434917,65.602076


### Key Findings — Análise 5

**1. Auto cresce mais devagar que o mercado combinado: CAGR de 7,6% a.a. vs. 10,5% do total (Análise 1)**
É o produto de crescimento mais lento entre os 3 do portfólio Youse, puxando o CAGR do mercado total para baixo.

**2. Share caiu de 83,3% (2016) para 65,6% (2025) — perda de 17,7 p.p. em 9 anos**
Auto continua sendo o maior produto, mas perdeu peso relativo de forma consistente, à medida que Vida e Residencial cresceram mais rápido.

**3. Sinistralidade oscila sem tendência clara de melhora estrutural**
O LR variou entre 57,7% (2023, melhor ano) e 69,4% (2026 YTD, pior ano) sem trajetória decrescente sustentada — diferente do padrão de queda do mercado total (Análise 1) e de Vida (Análise 13).

**4. 2026 YTD mostra sinistralidade subindo para o pior nível da série — ponto de atenção**
Reverte a melhora observada em 2023, num movimento que merece acompanhamento nos próximos meses.

> **Gráfico ThinkCell sugerido:** Combo chart (barras prêmio+sinistro em R$B + linha de LR), com anotação da queda de share de mercado ao longo do período.

## Análise 6 — Auto: Drill-Down Geográfico (Rolling 12M: mai/2025–abr/2026)

> **Subtítulo do slide:** SP concentra 41% do mercado de Auto, mas o crescimento mais rápido vem do Nordeste

Composição geográfica do prêmio direto de Auto: macro-região (comparável à Análise 2) e Top 10 UF, com crescimento YoY (rolling atual vs. rolling anterior).

In [15]:
total_produto = dfp_rolling['premio_dir'].sum()

# Por macro-região
a6_regiao = (
    dfp_rolling
    .groupby('macro_regiao', as_index=False)['premio_dir'].sum()
    .sort_values('premio_dir', ascending=False)
)
a6_regiao['share_pct'] = a6_regiao['premio_dir'] / total_produto * 100
a6_regiao['premio_B']  = a6_regiao['premio_dir'] / 1e9

# Top 10 UF com crescimento YoY
a6_uf_atual = dfp_rolling.groupby('UF', as_index=False)['premio_dir'].sum().rename(columns={'premio_dir': 'premio_atual'})
a6_uf_prev  = dfp_rolling_prev.groupby('UF', as_index=False)['premio_dir'].sum().rename(columns={'premio_dir': 'premio_prev'})
a6_uf = a6_uf_atual.merge(a6_uf_prev, on='UF', how='left').fillna({'premio_prev': 0})
a6_uf['share_pct'] = a6_uf['premio_atual'] / total_produto * 100
a6_uf['crescimento_pct'] = np.where(
    a6_uf['premio_prev'] > 0, (a6_uf['premio_atual'] / a6_uf['premio_prev'] - 1) * 100, np.nan
)
a6_uf['premio_B'] = a6_uf['premio_atual'] / 1e9
a6_uf = a6_uf.sort_values('premio_atual', ascending=False).reset_index(drop=True)
a6_uf['share_acum'] = a6_uf['share_pct'].cumsum()
a6_uf_top10 = a6_uf.head(10)

print(f"{produto_sel} — Prêmio Direto Total (mai/25–abr/26): R$ {total_produto/1e9:.2f}B\n")

print("Por Macro-Região:")
print(f"{'Região':<15}{'Prêmio (R$B)':>14}{'Share':>8}")
print("-" * 38)
for _, r in a6_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['premio_B']:>12.2f}B{r['share_pct']:>7.1f}%")

print("\nTop 10 UF (com crescimento YoY vs. mai/24-abr/25):")
print(f"{'UF':<5}{'Prêmio (R$B)':>13}{'Share':>8}{'Acum.':>8}{'Cresc. YoY':>13}")
print("-" * 48)
for _, r in a6_uf_top10.iterrows():
    cresc = f"{r['crescimento_pct']:.1f}%" if pd.notna(r['crescimento_pct']) else "n/d"
    print(f"{r['UF']:<5}{r['premio_B']:>11.2f}B{r['share_pct']:>7.1f}%{r['share_acum']:>7.1f}%{cresc:>13}")

print("\n--- Exportável ThinkCell (por região) ---")
display(a6_regiao[['macro_regiao','premio_B','share_pct']].rename(columns={'macro_regiao':'Regiao','premio_B':'Premio_B','share_pct':'Share_pct'}))

print("\n--- Exportável ThinkCell (Top 10 UF) ---")
display(a6_uf_top10[['UF','premio_B','share_pct','share_acum','crescimento_pct']].rename(
    columns={'premio_B':'Premio_B','share_pct':'Share_pct','share_acum':'Share_Acumulado','crescimento_pct':'Crescimento_YoY_pct'}))

Auto — Prêmio Direto Total (mai/25–abr/26): R$ 62.58B

Por Macro-Região:
Região           Prêmio (R$B)   Share
--------------------------------------
Sudeste               36.35B   58.1%
Sul                   12.92B   20.6%
Nordeste               6.83B   10.9%
Centro-Oeste           5.03B    8.0%
Norte                  1.45B    2.3%

Top 10 UF (com crescimento YoY vs. mai/24-abr/25):
UF    Prêmio (R$B)   Share   Acum.   Cresc. YoY
------------------------------------------------
SP         25.49B   40.7%   40.7%         5.1%
MG          5.68B    9.1%   49.8%         9.6%
PR          4.94B    7.9%   57.7%         7.0%
RS          4.42B    7.1%   64.8%         4.7%
RJ          3.98B    6.4%   71.1%         6.1%
SC          3.55B    5.7%   76.8%         8.4%
BA          2.08B    3.3%   80.1%        10.1%
GO          1.99B    3.2%   83.3%         7.9%
PE          1.47B    2.4%   85.6%        12.2%
MT          1.25B    2.0%   87.6%         4.5%

--- Exportável ThinkCell (por região) ---


,Regiao,Premio_B,Share_pct
3,Sudeste,36.354641,58.088821
4,Sul,12.918680,20.641955
1,Nordeste,6.828624,10.911034
0,Centro-Oeste,5.034985,8.045090
2,Norte,1.447644,2.313100



--- Exportável ThinkCell (Top 10 UF) ---


,UF,Premio_B,Share_pct,Share_Acumulado,Crescimento_YoY_pct
0,SP,25.485078,40.721023,40.721023,5.139674
1,MG,5.679692,9.075228,49.796250,9.606509
2,PR,4.944072,7.899825,57.696076,6.950483
3,RS,4.420541,7.063308,64.759383,4.746594
4,RJ,3.976837,6.354341,71.113724,6.051468
5,SC,3.554067,5.678822,76.792547,8.408367
6,BA,2.075616,3.316497,80.109044,10.068891
7,GO,1.990566,3.180601,83.289645,7.886890
8,PE,1.472826,2.353337,85.642982,12.199669
9,MT,1.247651,1.993544,87.636527,4.545419


### Key Findings — Análise 6

**1. SP concentra 40,7% do mercado de Auto sozinho — mais que MG+PR+RS somados (24,1%)**
O mercado de Auto é fortemente ancorado em um único estado, mais concentrado que a distribuição por macro-região sugere à primeira vista.

**2. Apesar da concentração em SP, o crescimento mais acelerado vem do Nordeste: PE (+12,2%) e BA (+10,1%)**
Mesmo com share pequeno (2,4% e 3,3%), esses estados crescem bem acima da média — sinal de que o Nordeste está ganhando tração em Auto, ainda que a partir de uma base pequena.

**3. Top 10 UF concentram 87,6% do mercado — a cauda longa de estados é irrelevante**
Praticamente todo o volume de Auto está nos 10 maiores estados; os 17 demais dividem menos de 13% do mercado.

**4. SP tem o crescimento mais lento entre o Top 10 (+5,1%) — sinal de maturidade no principal mercado**
Enquanto os estados menores crescem de dois dígitos, o mercado líder já dá sinais de estabilização.

> **Gráfico ThinkCell sugerido:** Barra horizontal por macro-região + ranking de Top 10 UF com destaque de crescimento YoY (cores por faixa de crescimento).

## Análise 7 — Auto: Sinistralidade por Região e Tendência Anual do LR

> **Subtítulo do slide:** Sinistralidade de Auto atinge o pior nível da série em 2026, com Centro-Oeste e Sul acima de 69%

Loss ratio de Auto por macro-região e Top 10 UF (rolling 12M), e evolução do loss ratio anual do produto (2016–2026).

In [16]:
tot_p_produto = dfp_rolling['premio_dir'].sum()
tot_s_produto = dfp_rolling['sin_dir'].sum()
lr_produto    = tot_s_produto / tot_p_produto * 100

# Por macro-região
a7_regiao = (
    dfp_rolling
    .groupby('macro_regiao', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a7_regiao['loss_ratio_pct'] = a7_regiao['sin_dir'] / a7_regiao['premio_dir'] * 100
a7_regiao['premio_B']       = a7_regiao['premio_dir'] / 1e9
a7_regiao['sinistro_B']     = a7_regiao['sin_dir']   / 1e9
a7_regiao = a7_regiao.sort_values('loss_ratio_pct', ascending=False)

# Top 10 UF por prêmio, com LR
a7_uf = (
    dfp_rolling
    .groupby('UF', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a7_uf['loss_ratio_pct'] = a7_uf['sin_dir'] / a7_uf['premio_dir'] * 100
a7_uf['premio_B']       = a7_uf['premio_dir'] / 1e9
a7_uf['sinistro_B']     = a7_uf['sin_dir']   / 1e9
a7_uf = a7_uf.sort_values('premio_dir', ascending=False).head(10)

# Tendência anual do LR (2016–2026), reaproveitando a5 (Análise 5)
a7_tendencia = a5[['ano','lr_pct']].copy()

print(f"{produto_sel} — Loss Ratio Total (mai/25–abr/26): {lr_produto:.1f}%\n")

print("Por Macro-Região:")
print(f"{'Região':<15}{'LR':>8}{'Prêmio(B)':>12}{'Sinistro(B)':>13}")
print("-" * 50)
for _, r in a7_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['loss_ratio_pct']:>7.1f}%{r['premio_B']:>10.2f}B{r['sinistro_B']:>11.2f}B")

print("\nTendência do Loss Ratio anual:")
print(f"{'Ano':<6}{'LR':>8}")
print("-" * 16)
for _, r in a7_tendencia.iterrows():
    ytd = " (YTD)" if r['ano'] == 2026 else ""
    print(f"{int(r['ano']):<6}{r['lr_pct']:>7.1f}%{ytd}")

print("\n--- Exportável ThinkCell (por região) ---")
display(a7_regiao[['macro_regiao','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'macro_regiao':'Regiao','loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

print("\n--- Exportável ThinkCell (Top 10 UF) ---")
display(a7_uf[['UF','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

print("\n--- Exportável ThinkCell (tendência anual do LR) ---")
display(a7_tendencia.rename(columns={'ano':'Ano','lr_pct':'LossRatio_pct'}))

Auto — Loss Ratio Total (mai/25–abr/26): 65.2%

Por Macro-Região:
Região               LR   Prêmio(B)  Sinistro(B)
--------------------------------------------------
Centro-Oeste      70.2%      5.03B       3.54B
Sul               69.6%     12.92B       8.98B
Norte             66.7%      1.45B       0.97B
Nordeste          64.5%      6.83B       4.41B
Sudeste           63.1%     36.35B      22.93B

Tendência do Loss Ratio anual:
Ano         LR
----------------
2016     68.9%
2017     65.4%
2018     62.1%
2019     64.1%
2020     58.1%
2021     69.0%
2022     66.3%
2023     57.7%
2024     63.9%
2025     64.4%
2026     69.4% (YTD)

--- Exportável ThinkCell (por região) ---


,Regiao,LossRatio_pct,Premio_B,Sinistro_B
0,Centro-Oeste,70.233047,5.034985,3.536223
4,Sul,69.550331,12.918680,8.984985
2,Norte,66.736061,1.447644,0.966100
1,Nordeste,64.513103,6.828624,4.405357
3,Sudeste,63.062549,36.354641,22.926163



--- Exportável ThinkCell (Top 10 UF) ---


,UF,LossRatio_pct,Premio_B,Sinistro_B
25,SP,61.299887,25.485078,15.622324
10,MG,70.483581,5.679692,4.003251
17,PR,69.412898,4.944072,3.431824
22,RS,68.702640,4.420541,3.037028
18,RJ,62.080900,3.976837,2.468856
23,SC,70.795873,3.554067,2.516133
4,BA,70.182292,2.075616,1.456715
8,GO,71.573173,1.990566,1.424711
15,PE,58.344898,1.472826,0.859319
12,MT,70.826746,1.247651,0.883671



--- Exportável ThinkCell (tendência anual do LR) ---


,Ano,LossRatio_pct
0,2016,68.862893
1,2017,65.427549
2,2018,62.137290
3,2019,64.132394
4,2020,58.083075
5,2021,68.954950
6,2022,66.302825
7,2023,57.672505
8,2024,63.945075
9,2025,64.434917


### Key Findings — Análise 7

**1. SP e Sudeste têm o menor LR, apesar de concentrarem 58% do prêmio**
SP tem LR de 61,3% e o Sudeste 63,1% — abaixo da média nacional de Auto (65,2%). Escala e maturidade do mercado parecem trazer vantagem de seleção de risco, não o contrário.

**2. Centro-Oeste e Sul têm sinistralidade acima de 69%, bem acima da média**
São as regiões de maior risco em Auto fora do eixo SP — crescer nessas praças significa assumir uma carteira estruturalmente mais arriscada.

**3. Pernambuco combina o menor LR do Top 10 UF (58,3%) com o maior crescimento (Análise 6, +12,2%)**
Uma combinação rara e valiosa: expansão geográfica com risco controlado — PE se destaca como praça de oportunidade em Auto.

**4. Sem melhora estrutural na sinistralidade: 2026 YTD (69,4%) é o pior nível da série 2016-2026**
Reverte a melhora observada em 2023 (57,7%, o melhor ano) — ponto de atenção para o mercado de Auto como um todo.

> **Gráfico ThinkCell sugerido:** Barra horizontal (LR por região/UF) + linha de tendência do LR anual, com marcador no pico de 2026 YTD.

## Análise 8 — Auto: Ranking de Competidores no Produto (Rolling 12M: mai/2025–abr/2026)

> **Subtítulo do slide:** Porto Seguro lidera Auto com 26% de share e a melhor sinistralidade entre os líderes

Top 10 seguradoras por prêmio direto de Auto, com classificação **especialista vs. generalista** (com base no mix de portfólio consolidado dos 3 produtos, calculado na Análise 4) e a posição isolada da Youse neste produto.

In [17]:
a8_cia = (
    dfp_rolling
    .groupby('coenti', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a8_cia = a8_cia.merge(ses_cias[['Coenti','Noenti']], left_on='coenti', right_on='Coenti', how='left')
a8_cia['noenti']         = a8_cia['Noenti'].fillna('Empresa ' + a8_cia['coenti'].astype(str))
a8_cia['loss_ratio_pct'] = a8_cia['sin_dir'] / a8_cia['premio_dir'] * 100
a8_cia['premio_B']       = a8_cia['premio_dir'] / 1e9
a8_cia = a8_cia.sort_values('premio_dir', ascending=False).reset_index(drop=True)

tot_mercado_produto = a8_cia['premio_dir'].sum()
n_empresas_produto  = len(a8_cia)

# Merge com o consolidado (3 produtos, Análise 4) para classificar especialista vs. generalista
a8_export = a8_cia[['coenti','noenti','premio_dir','premio_B','loss_ratio_pct']].head(10).copy()
a8_export = a8_export.merge(
    a4_cia[['coenti','premio_dir']].rename(columns={'premio_dir':'premio_total_3produtos'}),
    on='coenti', how='left'
)
a8_export['premio_total_3produtos'] = a8_export['premio_total_3produtos'].fillna(a8_export['premio_dir'])
a8_export['pct_portfolio']  = a8_export['premio_dir'] / a8_export['premio_total_3produtos'] * 100
a8_export['perfil']         = np.where(a8_export['pct_portfolio'] >= 70, 'Especialista', 'Generalista')
a8_export['share_pct']      = a8_export['premio_B'] / (tot_mercado_produto / 1e9) * 100
a8_export['share_acum']     = a8_export['share_pct'].cumsum()
top10_share_produto = a8_export['share_pct'].sum()

print(f"{produto_sel} — Ranking de Competidores (mai/2025–abr/2026)")
print(f"Mercado do produto: R$ {tot_mercado_produto/1e9:.2f}B | {n_empresas_produto} empresas ativas")
print(f"\n{'#':<3}{'Empresa':<40}{'Prêmio':>9}{'Share':>7}{'Acum.':>7}{'LR':>7}  Perfil (% portfólio)")
print("-" * 95)
for i, (_, r) in enumerate(a8_export.iterrows(), 1):
    nome = r['noenti'][:39]
    print(f"{i:<3}{nome:<40}{r['premio_B']:>7.2f}B{r['share_pct']:>6.1f}%{r['share_acum']:>6.1f}%"
          f"{r['loss_ratio_pct']:>6.1f}%  {r['perfil']} ({r['pct_portfolio']:.0f}%)")

print(f"\n  >> Top 10 concentra {top10_share_produto:.1f}% do mercado de {produto_sel}")

display(
    a8_export[['noenti','premio_B','share_pct','share_acum','loss_ratio_pct','pct_portfolio','perfil']]
    .rename(columns={
        'noenti':'Empresa','premio_B':'Premio_B','share_pct':'Share_pct',
        'share_acum':'Share_Acumulado','loss_ratio_pct':'LossRatio_pct',
        'pct_portfolio':'Pct_Portfolio_no_Produto','perfil':'Perfil'
    })
    .reset_index(drop=True)
)

# Posição isolada da Youse no produto
youse_prem_p = dfp_rolling[dfp_rolling['coenti'] == YOUSE_COENTI]['premio_dir'].sum()
youse_sin_p  = dfp_rolling[dfp_rolling['coenti'] == YOUSE_COENTI]['sin_dir'].sum()
youse_B_p    = youse_prem_p / 1e9
youse_shr_p  = youse_B_p / (tot_mercado_produto / 1e9) * 100
youse_rank_idx_p = a8_cia[a8_cia['coenti'] == YOUSE_COENTI].index
youse_rank_p = int(youse_rank_idx_p[0]) + 1 if len(youse_rank_idx_p) > 0 else None

print(f"\n{'='*60}")
print(f"Posição da Youse em {produto_sel} (mai/2025–abr/2026)")
print(f"{'='*60}")
if youse_prem_p > 0:
    youse_lr_p = youse_sin_p / youse_prem_p * 100
    rank_str = f"#{youse_rank_p}" if youse_rank_p else "n/d"
    print(f"Prêmio: R$ {youse_B_p:.3f}B | Share: {youse_shr_p:.2f}% | LR: {youse_lr_p:.1f}% | Rank: {rank_str} de {n_empresas_produto}")
else:
    print(f"Youse (coenti {YOUSE_COENTI}) não registra prêmio direto individualizado em {produto_sel} nesta base.")
    print("Consistente com a Análise 4: as operações da Youse são reportadas dentro do grupo Caixa Seguridade na SUSEP.")

Auto — Ranking de Competidores (mai/2025–abr/2026)
Mercado do produto: R$ 62.58B | 57 empresas ativas

#  Empresa                                    Prêmio  Share  Acum.     LR  Perfil (% portfólio)
-----------------------------------------------------------------------------------------------
1  PORTO SEGURO COMPANHIA DE SEGUROS GERAI   16.24B  26.0%  26.0%  53.5%  Especialista (87%)
2  TOKIO MARINE SEGURADORA S.A.               9.04B  14.4%  40.4%  63.1%  Especialista (91%)
3  ALLIANZ SEGUROS S.A.                       8.89B  14.2%  54.6%  69.3%  Especialista (91%)
4  BRADESCO AUTO/RE COMPANHIA DE SEGUROS      6.99B  11.2%  65.8%  71.6%  Especialista (86%)
5  YELUM SEGUROS S.A.                         5.76B   9.2%  75.0%  71.8%  Especialista (94%)
6  HDI SEGUROS S.A.                           4.49B   7.2%  82.1%  73.7%  Especialista (89%)
7  MAPFRE SEGUROS GERAIS S.A.                 3.26B   5.2%  87.3%  53.2%  Especialista (92%)
8  SUHAI SEGURADORA S.A.                      1.98B   

,Empresa,Premio_B,Share_pct,Share_Acumulado,LossRatio_pct,Pct_Portfolio_no_Produto,Perfil
0,PORTO SEGURO COMPANHIA DE SEGUROS GERAIS,16.241765,25.951706,25.951706,53.528699,87.476112,Especialista
1,TOKIO MARINE SEGURADORA S.A.,9.038451,14.441979,40.393685,63.121032,91.274420,Especialista
2,ALLIANZ SEGUROS S.A.,8.894837,14.212507,54.606192,69.349786,90.523036,Especialista
3,BRADESCO AUTO/RE COMPANHIA DE SEGUROS,6.985663,11.161957,65.768150,71.557740,85.870687,Especialista
4,YELUM SEGUROS S.A.,5.756308,9.197647,74.965796,71.761815,94.162134,Especialista
5,HDI SEGUROS S.A.,4.489672,7.173768,82.139564,73.703343,88.811231,Especialista
6,MAPFRE SEGUROS GERAIS S.A.,3.256045,5.202633,87.342197,53.202198,91.740666,Especialista
7,SUHAI SEGURADORA S.A.,1.976675,3.158406,90.500603,64.096014,99.999750,Especialista
8,ZURICH MINAS BRASIL SEGUROS S.A.,1.602963,2.561275,93.061878,71.502930,89.450604,Especialista
9,SANTANDER AUTO S.A.,0.706529,1.128918,94.190796,17.480861,100.000000,Especialista



Posição da Youse em Auto (mai/2025–abr/2026)
Youse (coenti 1121) não registra prêmio direto individualizado em Auto nesta base.
Consistente com a Análise 4: as operações da Youse são reportadas dentro do grupo Caixa Seguridade na SUSEP.


### Key Findings — Análise 8

**1. Porto Seguro lidera com 26,0% de share e o menor LR entre os líderes (53,5%)**
Uma combinação de escala e eficiência de subscrição — mais que o dobro de share do segundo colocado (Tokio Marine, 14,4%), com sinistralidade abaixo da média do produto (65,2%).

**2. Mercado hiperconcentrado: Top 10 = 94,2% do total**
Mais concentrado que o ranking consolidado dos 3 produtos (82,9%, Análise 4) — em Auto, praticamente não existe cauda longa relevante.

**3. Todos os Top 10 são "especialistas" em Auto — mesmo os grandes grupos diversificados**
Todos têm mais de 85% do próprio portfólio consolidado concentrado em Auto. Diferente de Residencial e Vida (onde aparecem generalistas puros), em Auto nenhum player relevante tem folga real de diversificação.

**4. Santander Auto é o outlier: LR de apenas 17,5%, muito abaixo da média**
Um modelo de nicho bancário com seleção de risco muito mais apurada que o resto do mercado — referência direta de benchmarking de subscrição para a Youse.

> **Nota de dados:** Youse (coenti 1121) não registra prêmio direto individualizado em Auto nesta base — mesma limitação documentada na Análise 4 (operação reportada dentro do grupo Caixa Seguridade).

> **Gráfico ThinkCell sugerido:** Barra horizontal rankeada por prêmio, com marcador de perfil (especialista/generalista) e linha de share acumulado.

## Análise 9 — Residencial: Evolução Histórica do Produto (2016–2026)

> **Subtítulo do slide:** Residencial cresce acima do mercado (11,9% a.a.) com sinistralidade estável entre 21% e 27%

Drill down da Análise 1 para o produto **Residencial**: prêmio e sinistro diretos por ano, loss ratio, CAGR e share do produto sobre o mercado total Youse.

In [18]:
produto_sel = 'Residencial'
dfp_anual = df_anual[df_anual['produto'] == produto_sel]
dfp_rolling = df_rolling[df_rolling['produto'] == produto_sel]
dfp_rolling_prev = df_rolling_prev[df_rolling_prev['produto'] == produto_sel]

a9 = (
    dfp_anual
    .groupby('ano', as_index=False)
    .agg(premio_dir=('premio_dir', 'sum'), sin_dir=('sin_dir', 'sum'))
)
a9['loss_ratio'] = a9['sin_dir'] / a9['premio_dir']
a9['premio_B']   = a9['premio_dir'] / 1e9
a9['sinistro_B'] = a9['sin_dir']   / 1e9
a9['lr_pct']     = a9['loss_ratio'] * 100
a9['share_mercado_pct'] = a9['ano'].map(total_ano_mercado)
a9['share_mercado_pct'] = a9['premio_dir'] / a9['share_mercado_pct'] * 100

# CAGR 2016–2025 (anos completos)
p2016 = a9.loc[a9['ano'] == 2016, 'premio_dir'].values[0]
p2025 = a9.loc[a9['ano'] == 2025, 'premio_dir'].values[0]
cagr  = (p2025 / p2016) ** (1/9) - 1

s2016    = a9.loc[a9['ano'] == 2016, 'sin_dir'].values[0]
s2025    = a9.loc[a9['ano'] == 2025, 'sin_dir'].values[0]
cagr_sin = (s2025 / s2016) ** (1/9) - 1

print(f"{produto_sel} — Prêmio, Sinistro e Share de Mercado (2016–2026)")
print(f"{'Ano':<6}{'Prêmio (R$B)':>14}{'Sinistro (R$B)':>16}{'Loss Ratio':>12}{'Share Merc.':>13}")
print("-" * 63)
for _, r in a9.iterrows():
    ytd = " (YTD)" if r['ano'] == 2026 else ""
    print(f"{int(r['ano']):<6}{r['premio_B']:>12.1f}B{r['sinistro_B']:>14.1f}B{r['lr_pct']:>11.1f}%{r['share_mercado_pct']:>12.1f}%{ytd}")

print(f"\nCAGR Prêmio 2016–2025:   {cagr*100:.1f}% a.a.")
print(f"CAGR Sinistro 2016–2025: {cagr_sin*100:.1f}% a.a.")
print(f"Loss Ratio 2025: {a9.loc[a9['ano']==2025,'lr_pct'].values[0]:.1f}%")
print(f"Share de mercado: {a9.loc[a9['ano']==2016,'share_mercado_pct'].values[0]:.1f}% (2016) -> {a9.loc[a9['ano']==2025,'share_mercado_pct'].values[0]:.1f}% (2025)")

# DataFrame para ThinkCell (exportável)
a9_export = a9[a9['ano'] <= 2025][['ano','premio_B','sinistro_B','lr_pct','share_mercado_pct']].copy()
a9_export.columns = ['Ano','Premio_B','Sinistro_B','LossRatio_pct','Share_Mercado_pct']
a9_export

Residencial — Prêmio, Sinistro e Share de Mercado (2016–2026)
Ano     Prêmio (R$B)  Sinistro (R$B)  Loss Ratio  Share Merc.
---------------------------------------------------------------
2016           2.9B           0.8B       26.7%         7.5%
2017           3.1B           0.8B       24.5%         7.5%
2018           3.5B           0.7B       21.4%         7.8%
2019           3.7B           0.8B       22.5%         7.7%
2020           3.9B           1.0B       26.5%         8.0%
2021           4.4B           1.1B       24.7%         8.0%
2022           5.1B           1.3B       26.3%         7.2%
2023           5.9B           1.6B       27.1%         7.5%
2024           7.1B           1.7B       23.6%         8.2%
2025           7.9B           1.8B       22.8%         8.4%
2026           2.7B           0.6B       24.0%         8.5% (YTD)

CAGR Prêmio 2016–2025:   11.9% a.a.
CAGR Sinistro 2016–2025: 10.0% a.a.
Loss Ratio 2025: 22.8%
Share de mercado: 7.5% (2016) -> 8.4% (2025)


,Ano,Premio_B,Sinistro_B,LossRatio_pct,Share_Mercado_pct
0,2016,2.865351,0.763685,26.652410,7.526158
1,2017,3.076571,0.755010,24.540640,7.502422
2,2018,3.460393,0.740108,21.387978,7.811369
3,2019,3.660312,0.824705,22.531003,7.742353
4,2020,3.891189,1.032261,26.528169,8.020132
5,2021,4.371021,1.077559,24.652352,8.001892
6,2022,5.050440,1.327175,26.278393,7.157701
7,2023,5.920263,1.604801,27.106912,7.493700
8,2024,7.052030,1.665099,23.611632,8.236044
9,2025,7.887256,1.800960,22.833791,8.433976


### Key Findings — Análise 9

**1. Residencial cresce acima do mercado total: CAGR de 11,9% vs. 10,5% do consolidado (Análise 1)**
O produto ganhou share de mercado, passando de 7,5% (2016) para 8,4% (2025) — crescimento consistente, ainda que seja o menor produto em volume absoluto.

**2. LR estrutural baixo e estável, entre 21% e 27% ao longo de 9 anos**
Sem grandes saltos ou reversões — o perfil de risco mais previsível dos 3 produtos Youse.

**3. Menor produto em volume absoluto (R$ 7,9B em 2025), mas com o crescimento de sinistralidade mais controlado**
O CAGR de sinistro (10,0%) fica muito próximo do CAGR de prêmio (11,9%) — crescimento sustentável, sem sinais de deterioração de risco associada à expansão.

> **Gráfico ThinkCell sugerido:** Combo chart (barras prêmio+sinistro em R$B + linha de LR), com bracket de CAGR 2016-2025.

## Análise 10 — Residencial: Drill-Down Geográfico (Rolling 12M: mai/2025–abr/2026)

> **Subtítulo do slide:** Nordeste e Centro-Oeste puxam o crescimento de Residencial, enquanto o DF recua 17%

Composição geográfica do prêmio direto de Residencial: macro-região e Top 10 UF, com crescimento YoY.

In [19]:
total_produto = dfp_rolling['premio_dir'].sum()

# Por macro-região
a10_regiao = (
    dfp_rolling
    .groupby('macro_regiao', as_index=False)['premio_dir'].sum()
    .sort_values('premio_dir', ascending=False)
)
a10_regiao['share_pct'] = a10_regiao['premio_dir'] / total_produto * 100
a10_regiao['premio_B']  = a10_regiao['premio_dir'] / 1e9

# Top 10 UF com crescimento YoY
a10_uf_atual = dfp_rolling.groupby('UF', as_index=False)['premio_dir'].sum().rename(columns={'premio_dir': 'premio_atual'})
a10_uf_prev  = dfp_rolling_prev.groupby('UF', as_index=False)['premio_dir'].sum().rename(columns={'premio_dir': 'premio_prev'})
a10_uf = a10_uf_atual.merge(a10_uf_prev, on='UF', how='left').fillna({'premio_prev': 0})
a10_uf['share_pct'] = a10_uf['premio_atual'] / total_produto * 100
a10_uf['crescimento_pct'] = np.where(
    a10_uf['premio_prev'] > 0, (a10_uf['premio_atual'] / a10_uf['premio_prev'] - 1) * 100, np.nan
)
a10_uf['premio_B'] = a10_uf['premio_atual'] / 1e9
a10_uf = a10_uf.sort_values('premio_atual', ascending=False).reset_index(drop=True)
a10_uf['share_acum'] = a10_uf['share_pct'].cumsum()
a10_uf_top10 = a10_uf.head(10)

print(f"{produto_sel} — Prêmio Direto Total (mai/25–abr/26): R$ {total_produto/1e9:.2f}B\n")

print("Por Macro-Região:")
print(f"{'Região':<15}{'Prêmio (R$B)':>14}{'Share':>8}")
print("-" * 38)
for _, r in a10_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['premio_B']:>12.2f}B{r['share_pct']:>7.1f}%")

print("\nTop 10 UF (com crescimento YoY vs. mai/24-abr/25):")
print(f"{'UF':<5}{'Prêmio (R$B)':>13}{'Share':>8}{'Acum.':>8}{'Cresc. YoY':>13}")
print("-" * 48)
for _, r in a10_uf_top10.iterrows():
    cresc = f"{r['crescimento_pct']:.1f}%" if pd.notna(r['crescimento_pct']) else "n/d"
    print(f"{r['UF']:<5}{r['premio_B']:>11.2f}B{r['share_pct']:>7.1f}%{r['share_acum']:>7.1f}%{cresc:>13}")

print("\n--- Exportável ThinkCell (por região) ---")
display(a10_regiao[['macro_regiao','premio_B','share_pct']].rename(columns={'macro_regiao':'Regiao','premio_B':'Premio_B','share_pct':'Share_pct'}))

print("\n--- Exportável ThinkCell (Top 10 UF) ---")
display(a10_uf_top10[['UF','premio_B','share_pct','share_acum','crescimento_pct']].rename(
    columns={'premio_B':'Premio_B','share_pct':'Share_pct','share_acum':'Share_Acumulado','crescimento_pct':'Crescimento_YoY_pct'}))

Residencial — Prêmio Direto Total (mai/25–abr/26): R$ 8.09B

Por Macro-Região:
Região           Prêmio (R$B)   Share
--------------------------------------
Sudeste                4.69B   57.9%
Sul                    2.04B   25.2%
Nordeste               0.67B    8.2%
Centro-Oeste           0.52B    6.4%
Norte                  0.18B    2.2%

Top 10 UF (com crescimento YoY vs. mai/24-abr/25):
UF    Prêmio (R$B)   Share   Acum.   Cresc. YoY
------------------------------------------------
SP          3.49B   43.1%   43.1%        10.5%
RS          0.81B   10.0%   53.1%        10.1%
PR          0.62B    7.6%   60.8%        17.0%
SC          0.61B    7.6%   68.3%        12.9%
RJ          0.58B    7.2%   75.5%         7.0%
MG          0.52B    6.4%   82.0%        10.5%
DF          0.18B    2.2%   84.2%       -16.6%
BA          0.18B    2.2%   86.4%        20.0%
GO          0.17B    2.1%   88.5%        18.7%
CE          0.13B    1.6%   90.0%        18.1%

--- Exportável ThinkCell (por região) -

,Regiao,Premio_B,Share_pct
3,Sudeste,4.688830,57.945366
4,Sul,2.037849,25.184089
1,Nordeste,0.667104,8.244185
0,Centro-Oeste,0.516130,6.378424
2,Norte,0.181899,2.247936



--- Exportável ThinkCell (Top 10 UF) ---


,UF,Premio_B,Share_pct,Share_Acumulado,Crescimento_YoY_pct
0,SP,3.490701,43.138686,43.138686,10.459747
1,RS,0.808482,9.991363,53.130049,10.127948
2,PR,0.617359,7.629428,60.759477,16.999959
3,SC,0.612008,7.563298,68.322775,12.935745
4,RJ,0.582180,7.194674,75.517449,6.950271
5,MG,0.520521,6.432689,81.950138,10.519172
6,DF,0.180698,2.233092,84.183230,-16.560941
7,BA,0.177181,2.189632,86.372862,20.001951
8,GO,0.168211,2.078779,88.451642,18.703456
9,CE,0.128626,1.589583,90.041225,18.097549


### Key Findings — Análise 10

**1. SP concentra 43,1% do mercado, mas o crescimento vem de fora do eixo tradicional**
Bahia (+20,0%), Goiás (+18,7%) e Ceará (+18,1%) crescem muito acima da média — o Nordeste e o Centro-Oeste emergem como frentes de expansão para o Residencial, mesmo com share ainda pequeno (2,1%-2,2% cada).

**2. DF é o único estado do Top 10 em retração: -16,6% YoY**
Um sinal de saturação ou perda de competitividade local que vale investigar — destoa completamente do padrão de crescimento dos demais estados do Centro-Oeste/Nordeste.

**3. Sul é a segunda maior região (25,2% do share), puxado por RS+PR+SC**
Os três estados do Sul somam praticamente o mesmo peso que o Sudeste fora de SP — uma região consolidada, mas de crescimento mais moderado que a fronteira Nordeste/Centro-Oeste.

> **Gráfico ThinkCell sugerido:** Barra horizontal por macro-região + ranking de Top 10 UF com destaque de crescimento YoY, sinalizando DF em vermelho (único em queda).

## Análise 11 — Residencial: Sinistralidade por Região e Tendência Anual do LR

> **Subtítulo do slide:** RS tem sinistralidade quase o dobro da média de Residencial, puxada por eventos climáticos

Loss ratio de Residencial por macro-região e Top 10 UF (rolling 12M), e evolução do loss ratio anual do produto (2016–2026).

In [26]:
tot_p_produto = dfp_rolling['premio_dir'].sum()
tot_s_produto = dfp_rolling['sin_dir'].sum()
lr_produto    = tot_s_produto / tot_p_produto * 100

# Por macro-região
a11_regiao = (
    dfp_rolling
    .groupby('macro_regiao', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a11_regiao['loss_ratio_pct'] = a11_regiao['sin_dir'] / a11_regiao['premio_dir'] * 100
a11_regiao['premio_B']       = a11_regiao['premio_dir'] / 1e9
a11_regiao['sinistro_B']     = a11_regiao['sin_dir']   / 1e9
a11_regiao = a11_regiao.sort_values('loss_ratio_pct', ascending=False)

# Top 10 UF por prêmio, com LR
a11_uf = (
    dfp_rolling
    .groupby('UF', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a11_uf['loss_ratio_pct'] = a11_uf['sin_dir'] / a11_uf['premio_dir'] * 100
a11_uf['premio_B']       = a11_uf['premio_dir'] / 1e9
a11_uf['sinistro_B']     = a11_uf['sin_dir']   / 1e9
a11_uf = a11_uf.sort_values('premio_dir', ascending=False).head(10)

# Tendência anual do LR (2016–2026), reaproveitando a9 (Análise 9)
a11_tendencia = a9[['ano','lr_pct']].copy()

print(f"{produto_sel} — Loss Ratio Total (mai/25–abr/26): {lr_produto:.1f}%\n")

print("Por Macro-Região:")
print(f"{'Região':<15}{'LR':>8}{'Prêmio(B)':>12}{'Sinistro(B)':>13}")
print("-" * 50)
for _, r in a11_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['loss_ratio_pct']:>7.1f}%{r['premio_B']:>10.2f}B{r['sinistro_B']:>11.2f}B")

print("\nTendência do Loss Ratio anual:")
print(f"{'Ano':<6}{'LR':>8}")
print("-" * 16)
for _, r in a11_tendencia.iterrows():
    ytd = " (YTD)" if r['ano'] == 2026 else ""
    print(f"{int(r['ano']):<6}{r['lr_pct']:>7.1f}%{ytd}")

print("\n--- Exportável ThinkCell (por região) ---")
display(a11_regiao[['macro_regiao','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'macro_regiao':'Regiao','loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

print("\n--- Exportável ThinkCell (Top 10 UF) ---")
display(a11_uf[['UF','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

print("\n--- Exportável ThinkCell (tendência anual do LR) ---")
display(a11_tendencia.rename(columns={'ano':'Ano','lr_pct':'LossRatio_pct'}))

Vida — Loss Ratio Total (mai/25–abr/26): 8.9%

Por Macro-Região:
Região               LR   Prêmio(B)  Sinistro(B)
--------------------------------------------------
Sul               10.1%      3.68B       0.37B
Sudeste            9.3%     16.84B       1.57B
Norte              8.5%      0.69B       0.06B
Centro-Oeste       7.7%      1.64B       0.13B
Nordeste           5.2%      2.36B       0.12B

Tendência do Loss Ratio anual:
Ano         LR
----------------
2016     26.7%
2017     24.5%
2018     21.4%
2019     22.5%
2020     26.5%
2021     24.7%
2022     26.3%
2023     27.1%
2024     23.6%
2025     22.8%
2026     24.0% (YTD)

--- Exportável ThinkCell (por região) ---


,Regiao,LossRatio_pct,Premio_B,Sinistro_B
4,Sul,10.061032,3.677375,0.369982
3,Sudeste,9.319460,16.835415,1.568970
2,Norte,8.504901,0.689726,0.058660
0,Centro-Oeste,7.703476,1.635172,0.125965
1,Nordeste,5.218251,2.356895,0.122989



--- Exportável ThinkCell (Top 10 UF) ---


,UF,LossRatio_pct,Premio_B,Sinistro_B
25,SP,7.583049,11.714466,0.888314
18,RJ,13.238195,2.919834,0.386533
10,MG,14.014995,1.868208,0.261829
17,PR,9.125796,1.503063,0.137166
22,RS,11.349348,1.294626,0.146932
23,SC,9.763007,0.879686,0.085884
4,BA,3.189608,0.711286,0.022687
15,PE,4.021964,0.591706,0.023798
8,GO,9.466479,0.567594,0.053731
5,CE,8.699738,0.470182,0.040905



--- Exportável ThinkCell (tendência anual do LR) ---


,Ano,LossRatio_pct
0,2016,26.652410
1,2017,24.540640
2,2018,21.387978
3,2019,22.531003
4,2020,26.528169
5,2021,24.652352
6,2022,26.278393
7,2023,27.106912
8,2024,23.611632
9,2025,22.833791


### Key Findings — Análise 11

**1. Sul tem LR muito acima da média nacional do produto — RS é o ponto de atenção**
O Sul tem LR de 32,0% (vs. média de 22,9% do produto), puxado por RS isoladamente com 40,7% — quase o dobro da média nacional. É um padrão consistente com maior exposição a eventos climáticos (enchentes) na região.

**2. Nordeste e Centro-Oeste têm risco estruturalmente mais baixo**
LR de 11,1% (Nordeste) e 14,8% (Centro-Oeste) — bem abaixo da média — reforçam o Residencial como produto de baixo risco fora do eixo Sul.

**3. O LR anual oscila numa faixa estreita (21,4%–27,1%) sem tendência clara de piora ou melhora**
Diferente de Auto (LR alto e volátil) e Vida (LR em queda estrutural), o Residencial é o produto mais previsível e estável dos três ao longo dos 9 anos analisados.

**4. RS cresce (10,1% do share, Análise 10) e tem quase o dobro da sinistralidade média — tensão entre crescimento e risco**
Expandir a base de clientes no RS significa assumir uma sinistralidade estruturalmente mais alta — vale considerar precificação regional diferenciada.

> **Gráfico ThinkCell sugerido:** Barra horizontal (LR por região/UF) + linha de tendência do LR anual, destacando RS como outlier de risco.

## Análise 12 — Residencial: Ranking de Competidores no Produto (Rolling 12M: mai/2025–abr/2026)

> **Subtítulo do slide:** Mix competitivo equilibrado em Residencial; XS3 se destaca com 14% de share e LR de apenas 3,4%

Top 10 seguradoras por prêmio direto de Residencial, com classificação **especialista vs. generalista** e a posição isolada da Youse neste produto.

In [21]:
a12_cia = (
    dfp_rolling
    .groupby('coenti', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a12_cia = a12_cia.merge(ses_cias[['Coenti','Noenti']], left_on='coenti', right_on='Coenti', how='left')
a12_cia['noenti']         = a12_cia['Noenti'].fillna('Empresa ' + a12_cia['coenti'].astype(str))
a12_cia['loss_ratio_pct'] = a12_cia['sin_dir'] / a12_cia['premio_dir'] * 100
a12_cia['premio_B']       = a12_cia['premio_dir'] / 1e9
a12_cia = a12_cia.sort_values('premio_dir', ascending=False).reset_index(drop=True)

tot_mercado_produto = a12_cia['premio_dir'].sum()
n_empresas_produto  = len(a12_cia)

# Merge com o consolidado (3 produtos, Análise 4) para classificar especialista vs. generalista
a12_export = a12_cia[['coenti','noenti','premio_dir','premio_B','loss_ratio_pct']].head(10).copy()
a12_export = a12_export.merge(
    a4_cia[['coenti','premio_dir']].rename(columns={'premio_dir':'premio_total_3produtos'}),
    on='coenti', how='left'
)
a12_export['premio_total_3produtos'] = a12_export['premio_total_3produtos'].fillna(a12_export['premio_dir'])
a12_export['pct_portfolio']  = a12_export['premio_dir'] / a12_export['premio_total_3produtos'] * 100
a12_export['perfil']         = np.where(a12_export['pct_portfolio'] >= 70, 'Especialista', 'Generalista')
a12_export['share_pct']      = a12_export['premio_B'] / (tot_mercado_produto / 1e9) * 100
a12_export['share_acum']     = a12_export['share_pct'].cumsum()
top10_share_produto = a12_export['share_pct'].sum()

print(f"{produto_sel} — Ranking de Competidores (mai/2025–abr/2026)")
print(f"Mercado do produto: R$ {tot_mercado_produto/1e9:.2f}B | {n_empresas_produto} empresas ativas")
print(f"\n{'#':<3}{'Empresa':<40}{'Prêmio':>9}{'Share':>7}{'Acum.':>7}{'LR':>7}  Perfil (% portfólio)")
print("-" * 95)
for i, (_, r) in enumerate(a12_export.iterrows(), 1):
    nome = r['noenti'][:39]
    print(f"{i:<3}{nome:<40}{r['premio_B']:>7.2f}B{r['share_pct']:>6.1f}%{r['share_acum']:>6.1f}%"
          f"{r['loss_ratio_pct']:>6.1f}%  {r['perfil']} ({r['pct_portfolio']:.0f}%)")

print(f"\n  >> Top 10 concentra {top10_share_produto:.1f}% do mercado de {produto_sel}")

display(
    a12_export[['noenti','premio_B','share_pct','share_acum','loss_ratio_pct','pct_portfolio','perfil']]
    .rename(columns={
        'noenti':'Empresa','premio_B':'Premio_B','share_pct':'Share_pct',
        'share_acum':'Share_Acumulado','loss_ratio_pct':'LossRatio_pct',
        'pct_portfolio':'Pct_Portfolio_no_Produto','perfil':'Perfil'
    })
    .reset_index(drop=True)
)

# Posição isolada da Youse no produto
youse_prem_p = dfp_rolling[dfp_rolling['coenti'] == YOUSE_COENTI]['premio_dir'].sum()
youse_sin_p  = dfp_rolling[dfp_rolling['coenti'] == YOUSE_COENTI]['sin_dir'].sum()
youse_B_p    = youse_prem_p / 1e9
youse_shr_p  = youse_B_p / (tot_mercado_produto / 1e9) * 100
youse_rank_idx_p = a12_cia[a12_cia['coenti'] == YOUSE_COENTI].index
youse_rank_p = int(youse_rank_idx_p[0]) + 1 if len(youse_rank_idx_p) > 0 else None

print(f"\n{'='*60}")
print(f"Posição da Youse em {produto_sel} (mai/2025–abr/2026)")
print(f"{'='*60}")
if youse_prem_p > 0:
    youse_lr_p = youse_sin_p / youse_prem_p * 100
    rank_str = f"#{youse_rank_p}" if youse_rank_p else "n/d"
    print(f"Prêmio: R$ {youse_B_p:.3f}B | Share: {youse_shr_p:.2f}% | LR: {youse_lr_p:.1f}% | Rank: {rank_str} de {n_empresas_produto}")
else:
    print(f"Youse (coenti {YOUSE_COENTI}) não registra prêmio direto individualizado em {produto_sel} nesta base.")
    print("Consistente com a Análise 4: as operações da Youse são reportadas dentro do grupo Caixa Seguridade na SUSEP.")

Residencial — Ranking de Competidores (mai/2025–abr/2026)


Mercado do produto: R$ 8.09B | 63 empresas ativas

#  Empresa                                    Prêmio  Share  Acum.     LR  Perfil (% portfólio)
-----------------------------------------------------------------------------------------------
1  PORTO SEGURO COMPANHIA DE SEGUROS GERAI    1.55B  19.1%  19.1%  30.0%  Generalista (8%)
2  BRADESCO AUTO/RE COMPANHIA DE SEGUROS      1.15B  14.2%  33.3%  16.9%  Generalista (14%)
3  XS3 SEGUROS S.A.                           1.10B  13.6%  46.9%   3.4%  Especialista (100%)
4  ALLIANZ SEGUROS S.A.                       0.78B   9.7%  56.6%  37.6%  Generalista (8%)
5  TOKIO MARINE SEGURADORA S.A.               0.72B   8.9%  65.6%  29.4%  Generalista (7%)
6  HDI SEGUROS S.A.                           0.54B   6.6%  72.2%  49.4%  Generalista (11%)
7  ALIANÇA DO BRASIL SEGUROS S.A              0.51B   6.3%  78.5%  17.0%  Especialista (100%)
8  ZURICH SANTANDER BRASIL SEGUROS S.A.       0.48B   5.9%  84.5%   7.5%  Especialista (100%)
9  CHUBB SEGUROS 

,Empresa,Premio_B,Share_pct,Share_Acumulado,LossRatio_pct,Pct_Portfolio_no_Produto,Perfil
0,PORTO SEGURO COMPANHIA DE SEGUROS GERAIS,1.546224,19.108507,19.108507,29.987278,8.327772,Generalista
1,BRADESCO AUTO/RE COMPANHIA DE SEGUROS,1.149433,14.204894,33.313401,16.853756,14.129313,Generalista
2,XS3 SEGUROS S.A.,1.103134,13.632721,46.946122,3.382837,100.000000,Especialista
3,ALLIANZ SEGUROS S.A.,0.783071,9.677327,56.623449,37.578725,7.969340,Generalista
4,TOKIO MARINE SEGURADORA S.A.,0.723844,8.945385,65.568834,29.406718,7.309706,Generalista
5,HDI SEGUROS S.A.,0.537036,6.636789,72.205623,49.383178,10.623242,Generalista
6,ALIANÇA DO BRASIL SEGUROS S.A,0.513163,6.341762,78.547386,16.967894,100.000000,Especialista
7,ZURICH SANTANDER BRASIL SEGUROS S.A.,0.479041,5.920074,84.467460,7.537152,100.000000,Especialista
8,CHUBB SEGUROS BRASIL S.A.,0.229799,2.839901,87.307361,12.550787,99.916948,Especialista
9,MAPFRE SEGUROS GERAIS S.A.,0.210883,2.606132,89.913493,30.311884,5.941741,Generalista



Posição da Youse em Residencial (mai/2025–abr/2026)
Youse (coenti 1121) não registra prêmio direto individualizado em Residencial nesta base.
Consistente com a Análise 4: as operações da Youse são reportadas dentro do grupo Caixa Seguridade na SUSEP.


### Key Findings — Análise 12

**1. Mix competitivo equilibrado entre generalistas e especialistas — diferente de Auto**
Porto Seguro, Bradesco, Allianz, Tokio Marine, HDI e Mapfre (generalistas, cada um com menos de 15% do próprio portfólio em Residencial) dividem o ranking com XS3 Seguros, Aliança do Brasil, Zurich Santander e Chubb (especialistas, 100% do portfólio em Residencial).

**2. XS3 Seguros é o destaque: 13,6% de share com LR de apenas 3,4%**
Uma seguradora 100% focada em Residencial operando com sinistralidade extremamente baixa — referência direta de benchmarking para o produto Youse Reside.

**3. O padrão "banco como canal de baixo risco" se repete**
Zurich Santander (LR 7,5%) tem um perfil de risco muito parecido com o do Santander Auto (Análise 8) — sugere que a originação via canal bancário seleciona sistematicamente clientes de menor risco, um padrão que se repete em produtos diferentes.

**4. HDI é o outlier de risco do Top 10: LR de 49,4%, mais que o dobro da média do produto**
Mesmo sendo generalista com participação pequena em Residencial (11% do portfólio), a HDI opera com sinistralidade muito acima dos pares.

> **Nota de dados:** Youse (coenti 1121) não registra prêmio direto individualizado em Residencial nesta base — mesma limitação documentada na Análise 4.

> **Gráfico ThinkCell sugerido:** Barra horizontal rankeada por prêmio, com marcador de perfil (especialista/generalista) e linha de share acumulado.

## Análise 13 — Vida: Evolução Histórica do Produto (2016–2026)

> **Subtítulo do slide:** Vida triplica o share de mercado (9% para 26%) com sinistralidade caindo à metade

Drill down da Análise 1 para o produto **Vida**: prêmio e sinistro diretos por ano, loss ratio, CAGR e share do produto sobre o mercado total Youse.

In [22]:
produto_sel = 'Vida'
dfp_anual = df_anual[df_anual['produto'] == produto_sel]
dfp_rolling = df_rolling[df_rolling['produto'] == produto_sel]
dfp_rolling_prev = df_rolling_prev[df_rolling_prev['produto'] == produto_sel]

a13 = (
    dfp_anual
    .groupby('ano', as_index=False)
    .agg(premio_dir=('premio_dir', 'sum'), sin_dir=('sin_dir', 'sum'))
)
a13['loss_ratio'] = a13['sin_dir'] / a13['premio_dir']
a13['premio_B']   = a13['premio_dir'] / 1e9
a13['sinistro_B'] = a13['sin_dir']   / 1e9
a13['lr_pct']     = a13['loss_ratio'] * 100
a13['share_mercado_pct'] = a13['ano'].map(total_ano_mercado)
a13['share_mercado_pct'] = a13['premio_dir'] / a13['share_mercado_pct'] * 100

# CAGR 2016–2025 (anos completos)
p2016 = a13.loc[a13['ano'] == 2016, 'premio_dir'].values[0]
p2025 = a13.loc[a13['ano'] == 2025, 'premio_dir'].values[0]
cagr  = (p2025 / p2016) ** (1/9) - 1

s2016    = a13.loc[a13['ano'] == 2016, 'sin_dir'].values[0]
s2025    = a13.loc[a13['ano'] == 2025, 'sin_dir'].values[0]
cagr_sin = (s2025 / s2016) ** (1/9) - 1

print(f"{produto_sel} — Prêmio, Sinistro e Share de Mercado (2016–2026)")
print(f"{'Ano':<6}{'Prêmio (R$B)':>14}{'Sinistro (R$B)':>16}{'Loss Ratio':>12}{'Share Merc.':>13}")
print("-" * 63)
for _, r in a13.iterrows():
    ytd = " (YTD)" if r['ano'] == 2026 else ""
    print(f"{int(r['ano']):<6}{r['premio_B']:>12.1f}B{r['sinistro_B']:>14.1f}B{r['lr_pct']:>11.1f}%{r['share_mercado_pct']:>12.1f}%{ytd}")

print(f"\nCAGR Prêmio 2016–2025:   {cagr*100:.1f}% a.a.")
print(f"CAGR Sinistro 2016–2025: {cagr_sin*100:.1f}% a.a.")
print(f"Loss Ratio 2025: {a13.loc[a13['ano']==2025,'lr_pct'].values[0]:.1f}%")
print(f"Share de mercado: {a13.loc[a13['ano']==2016,'share_mercado_pct'].values[0]:.1f}% (2016) -> {a13.loc[a13['ano']==2025,'share_mercado_pct'].values[0]:.1f}% (2025)")

# DataFrame para ThinkCell (exportável)
a13_export = a13[a13['ano'] <= 2025][['ano','premio_B','sinistro_B','lr_pct','share_mercado_pct']].copy()
a13_export.columns = ['Ano','Premio_B','Sinistro_B','LossRatio_pct','Share_Mercado_pct']
a13_export

Vida — Prêmio, Sinistro e Share de Mercado (2016–2026)
Ano     Prêmio (R$B)  Sinistro (R$B)  Loss Ratio  Share Merc.
---------------------------------------------------------------
2016           3.5B           0.6B       16.0%         9.2%
2017           4.1B           0.6B       15.5%        10.0%
2018           5.0B           0.7B       13.6%        11.3%
2019           7.7B           0.8B       10.7%        16.2%
2020           9.4B           1.1B       11.4%        19.3%
2021          11.9B           1.6B       13.5%        21.8%
2022          14.6B           1.6B       10.8%        20.6%
2023          17.4B           1.6B        9.3%        22.0%
2024          21.1B           1.9B        8.9%        24.6%
2025          24.3B           2.1B        8.5%        26.0%
2026           8.4B           0.8B        9.5%        26.9% (YTD)

CAGR Prêmio 2016–2025:   24.0% a.a.
CAGR Sinistro 2016–2025: 15.7% a.a.
Loss Ratio 2025: 8.5%
Share de mercado: 9.2% (2016) -> 26.0% (2025)


,Ano,Premio_B,Sinistro_B,LossRatio_pct,Share_Mercado_pct
0,2016,3.491463,0.557621,15.970978,9.170709
1,2017,4.108232,0.637063,15.506984,10.018195
2,2018,5.001026,0.681836,13.633929,11.289140
3,2019,7.667481,0.819308,10.685489,16.218386
4,2020,9.384268,1.071324,11.416173,19.341920
5,2021,11.934979,1.609168,13.482787,21.848995
6,2022,14.559856,1.577706,10.835999,20.634852
7,2023,17.388722,1.616989,9.299065,22.010148
8,2024,21.090311,1.886589,8.945289,24.631310
9,2025,24.280876,2.067386,8.514464,25.963948


### Key Findings — Análise 13

**1. Vida é o produto de crescimento mais explosivo: CAGR de 24,0% a.a. (2016–2025)**
Mais que o dobro do CAGR do mercado total (10,5%, Análise 1) e mais que o triplo do CAGR de Auto (7,6%, Análise 5).

**2. Share de mercado quase triplicou: de 9,2% (2016) para 26,0% (2025)**
É o maior ganho de participação entre os 3 produtos — Vida passou de um produto marginal para responder por mais de um quarto do mercado combinado Youse.

**3. Sinistralidade caiu estruturalmente à metade, mesmo com o crescimento acelerado**
O LR caiu de 16,0% para 8,5% no mesmo período — o forte crescimento de volume não veio acompanhado de deterioração de risco, ao contrário do que normalmente se esperaria de uma expansão rápida.

**4. 2026 YTD mostra leve alta no LR (9,5%), mas o produto segue o mais saudável do portfólio**
Mesmo com a leve reversão, o LR de Vida continua muito abaixo de Auto e Residencial.

> **Gráfico ThinkCell sugerido:** Combo chart (barras prêmio+sinistro em R$B + linha de LR), com bracket de CAGR e anotação do salto de share de mercado.

## Análise 14 — Vida: Drill-Down Geográfico (Rolling 12M: mai/2025–abr/2026)

> **Subtítulo do slide:** Vida é o produto mais concentrado no Sudeste (67%), mas cresce mais rápido fora dele

Composição geográfica do prêmio direto de Vida: macro-região e Top 10 UF, com crescimento YoY.

In [23]:
total_produto = dfp_rolling['premio_dir'].sum()

# Por macro-região
a14_regiao = (
    dfp_rolling
    .groupby('macro_regiao', as_index=False)['premio_dir'].sum()
    .sort_values('premio_dir', ascending=False)
)
a14_regiao['share_pct'] = a14_regiao['premio_dir'] / total_produto * 100
a14_regiao['premio_B']  = a14_regiao['premio_dir'] / 1e9

# Top 10 UF com crescimento YoY
a14_uf_atual = dfp_rolling.groupby('UF', as_index=False)['premio_dir'].sum().rename(columns={'premio_dir': 'premio_atual'})
a14_uf_prev  = dfp_rolling_prev.groupby('UF', as_index=False)['premio_dir'].sum().rename(columns={'premio_dir': 'premio_prev'})
a14_uf = a14_uf_atual.merge(a14_uf_prev, on='UF', how='left').fillna({'premio_prev': 0})
a14_uf['share_pct'] = a14_uf['premio_atual'] / total_produto * 100
a14_uf['crescimento_pct'] = np.where(
    a14_uf['premio_prev'] > 0, (a14_uf['premio_atual'] / a14_uf['premio_prev'] - 1) * 100, np.nan
)
a14_uf['premio_B'] = a14_uf['premio_atual'] / 1e9
a14_uf = a14_uf.sort_values('premio_atual', ascending=False).reset_index(drop=True)
a14_uf['share_acum'] = a14_uf['share_pct'].cumsum()
a14_uf_top10 = a14_uf.head(10)

print(f"{produto_sel} — Prêmio Direto Total (mai/25–abr/26): R$ {total_produto/1e9:.2f}B\n")

print("Por Macro-Região:")
print(f"{'Região':<15}{'Prêmio (R$B)':>14}{'Share':>8}")
print("-" * 38)
for _, r in a14_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['premio_B']:>12.2f}B{r['share_pct']:>7.1f}%")

print("\nTop 10 UF (com crescimento YoY vs. mai/24-abr/25):")
print(f"{'UF':<5}{'Prêmio (R$B)':>13}{'Share':>8}{'Acum.':>8}{'Cresc. YoY':>13}")
print("-" * 48)
for _, r in a14_uf_top10.iterrows():
    cresc = f"{r['crescimento_pct']:.1f}%" if pd.notna(r['crescimento_pct']) else "n/d"
    print(f"{r['UF']:<5}{r['premio_B']:>11.2f}B{r['share_pct']:>7.1f}%{r['share_acum']:>7.1f}%{cresc:>13}")

print("\n--- Exportável ThinkCell (por região) ---")
display(a14_regiao[['macro_regiao','premio_B','share_pct']].rename(columns={'macro_regiao':'Regiao','premio_B':'Premio_B','share_pct':'Share_pct'}))

print("\n--- Exportável ThinkCell (Top 10 UF) ---")
display(a14_uf_top10[['UF','premio_B','share_pct','share_acum','crescimento_pct']].rename(
    columns={'premio_B':'Premio_B','share_pct':'Share_pct','share_acum':'Share_Acumulado','crescimento_pct':'Crescimento_YoY_pct'}))

Vida — Prêmio Direto Total (mai/25–abr/26): R$ 25.19B

Por Macro-Região:
Região           Prêmio (R$B)   Share
--------------------------------------
Sudeste               16.84B   66.8%
Sul                    3.68B   14.6%
Nordeste               2.36B    9.4%
Centro-Oeste           1.64B    6.5%
Norte                  0.69B    2.7%

Top 10 UF (com crescimento YoY vs. mai/24-abr/25):
UF    Prêmio (R$B)   Share   Acum.   Cresc. YoY
------------------------------------------------
SP         11.71B   46.5%   46.5%        12.9%
RJ          2.92B   11.6%   58.1%        23.2%
MG          1.87B    7.4%   65.5%        10.6%
PR          1.50B    6.0%   71.5%        15.8%
RS          1.29B    5.1%   76.6%        14.0%
SC          0.88B    3.5%   80.1%        10.6%
BA          0.71B    2.8%   82.9%        18.8%
PE          0.59B    2.3%   85.3%        29.1%
GO          0.57B    2.3%   87.5%         7.5%
CE          0.47B    1.9%   89.4%        22.4%

--- Exportável ThinkCell (por região) ---


,Regiao,Premio_B,Share_pct
3,Sudeste,16.835415,66.821567
4,Sul,3.677375,14.595895
1,Nordeste,2.356895,9.354771
0,Centro-Oeste,1.635172,6.490173
2,Norte,0.689726,2.737595



--- Exportável ThinkCell (Top 10 UF) ---


,UF,Premio_B,Share_pct,Share_Acumulado,Crescimento_YoY_pct
0,SP,11.714466,46.495973,46.495973,12.857306
1,RJ,2.919834,11.589132,58.085105,23.173188
2,MG,1.868208,7.415119,65.500224,10.626433
3,PR,1.503063,5.965817,71.466041,15.800221
4,RS,1.294626,5.138510,76.604550,14.046216
5,SC,0.879686,3.491568,80.096118,10.635168
6,BA,0.711286,2.823172,82.919290,18.835865
7,PE,0.591706,2.348546,85.267836,29.148301
8,GO,0.567594,2.252842,87.520679,7.547157
9,CE,0.470182,1.866203,89.386882,22.428760


### Key Findings — Análise 14

**1. Vida é o produto mais concentrado geograficamente: 66,8% no Sudeste**
A concentração no Sudeste é ainda maior que em Auto (58,1%) e Residencial (57,9%) — o produto depende mais do eixo SP-RJ-MG do que os outros dois.

**2. O crescimento vem de fora do eixo tradicional: Pernambuco lidera com +29,1% YoY**
É o maior crescimento entre todos os cortes de UF analisados nos 3 produtos. Rio de Janeiro (+23,2%) e Ceará (+22,4%) também crescem bem acima da média — sinal de expansão geográfica ativa do produto Vida para fora do Sudeste tradicional.

**3. SP e RJ somam 58,1% do volume, mas crescem mais devagar que a fronteira**
SP cresce 12,9% e RJ 23,2% — ainda assim, os estados menores (PE, CE, BA) crescem mais rápido em termos relativos, típico de um produto em fase de expansão geográfica.

> **Gráfico ThinkCell sugerido:** Barra horizontal por macro-região + ranking de Top 10 UF com destaque de crescimento YoY (cores por faixa de crescimento).

## Análise 15 — Vida: Sinistralidade por Região e Tendência Anual do LR

> **Subtítulo do slide:** Vida tem a menor sinistralidade em todas as regiões e a queda mais consistente entre os 3 produtos

Loss ratio de Vida por macro-região e Top 10 UF (rolling 12M), e evolução do loss ratio anual do produto (2016–2026).

In [27]:
tot_p_produto = dfp_rolling['premio_dir'].sum()
tot_s_produto = dfp_rolling['sin_dir'].sum()
lr_produto    = tot_s_produto / tot_p_produto * 100

# Por macro-região
a15_regiao = (
    dfp_rolling
    .groupby('macro_regiao', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a15_regiao['loss_ratio_pct'] = a15_regiao['sin_dir'] / a15_regiao['premio_dir'] * 100
a15_regiao['premio_B']       = a15_regiao['premio_dir'] / 1e9
a15_regiao['sinistro_B']     = a15_regiao['sin_dir']   / 1e9
a15_regiao = a15_regiao.sort_values('loss_ratio_pct', ascending=False)

# Top 10 UF por prêmio, com LR
a15_uf = (
    dfp_rolling
    .groupby('UF', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a15_uf['loss_ratio_pct'] = a15_uf['sin_dir'] / a15_uf['premio_dir'] * 100
a15_uf['premio_B']       = a15_uf['premio_dir'] / 1e9
a15_uf['sinistro_B']     = a15_uf['sin_dir']   / 1e9
a15_uf = a15_uf.sort_values('premio_dir', ascending=False).head(10)

# Tendência anual do LR (2016–2026), reaproveitando a13 (Análise 13)
a15_tendencia = a13[['ano','lr_pct']].copy()

print(f"{produto_sel} — Loss Ratio Total (mai/25–abr/26): {lr_produto:.1f}%\n")

print("Por Macro-Região:")
print(f"{'Região':<15}{'LR':>8}{'Prêmio(B)':>12}{'Sinistro(B)':>13}")
print("-" * 50)
for _, r in a15_regiao.iterrows():
    print(f"{r['macro_regiao']:<15}{r['loss_ratio_pct']:>7.1f}%{r['premio_B']:>10.2f}B{r['sinistro_B']:>11.2f}B")

print("\nTop 10 UF (por prêmio):")
print(f"{'UF':<5}{'LR':>8}{'Prêmio(B)':>12}{'Sinistro(B)':>13}")
print("-" * 40)
for _, r in a15_uf.iterrows():
    print(f"{r['UF']:<5}{r['loss_ratio_pct']:>7.1f}%{r['premio_B']:>10.2f}B{r['sinistro_B']:>11.2f}B")

print("\nTendência do Loss Ratio anual:")
print(f"{'Ano':<6}{'LR':>8}")
print("-" * 16)
for _, r in a15_tendencia.iterrows():
    ytd = " (YTD)" if r['ano'] == 2026 else ""
    print(f"{int(r['ano']):<6}{r['lr_pct']:>7.1f}%{ytd}")

print("\n--- Exportável ThinkCell (por região) ---")
display(a15_regiao[['macro_regiao','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'macro_regiao':'Regiao','loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

print("\n--- Exportável ThinkCell (Top 10 UF) ---")
display(a15_uf[['UF','loss_ratio_pct','premio_B','sinistro_B']].rename(
    columns={'loss_ratio_pct':'LossRatio_pct','premio_B':'Premio_B','sinistro_B':'Sinistro_B'}))

print("\n--- Exportável ThinkCell (tendência anual do LR) ---")
display(a15_tendencia.rename(columns={'ano':'Ano','lr_pct':'LossRatio_pct'}))

Vida — Loss Ratio Total (mai/25–abr/26): 8.9%

Por Macro-Região:
Região               LR   Prêmio(B)  Sinistro(B)
--------------------------------------------------
Sul               10.1%      3.68B       0.37B
Sudeste            9.3%     16.84B       1.57B
Norte              8.5%      0.69B       0.06B
Centro-Oeste       7.7%      1.64B       0.13B
Nordeste           5.2%      2.36B       0.12B

Top 10 UF (por prêmio):
UF         LR   Prêmio(B)  Sinistro(B)
----------------------------------------
SP       7.6%     11.71B       0.89B
RJ      13.2%      2.92B       0.39B
MG      14.0%      1.87B       0.26B
PR       9.1%      1.50B       0.14B
RS      11.3%      1.29B       0.15B
SC       9.8%      0.88B       0.09B
BA       3.2%      0.71B       0.02B
PE       4.0%      0.59B       0.02B
GO       9.5%      0.57B       0.05B
CE       8.7%      0.47B       0.04B

Tendência do Loss Ratio anual:
Ano         LR
----------------
2016     16.0%
2017     15.5%
2018     13.6%
2019     10.7%
2

,Regiao,LossRatio_pct,Premio_B,Sinistro_B
4,Sul,10.061032,3.677375,0.369982
3,Sudeste,9.319460,16.835415,1.568970
2,Norte,8.504901,0.689726,0.058660
0,Centro-Oeste,7.703476,1.635172,0.125965
1,Nordeste,5.218251,2.356895,0.122989



--- Exportável ThinkCell (Top 10 UF) ---


,UF,LossRatio_pct,Premio_B,Sinistro_B
25,SP,7.583049,11.714466,0.888314
18,RJ,13.238195,2.919834,0.386533
10,MG,14.014995,1.868208,0.261829
17,PR,9.125796,1.503063,0.137166
22,RS,11.349348,1.294626,0.146932
23,SC,9.763007,0.879686,0.085884
4,BA,3.189608,0.711286,0.022687
15,PE,4.021964,0.591706,0.023798
8,GO,9.466479,0.567594,0.053731
5,CE,8.699738,0.470182,0.040905



--- Exportável ThinkCell (tendência anual do LR) ---


,Ano,LossRatio_pct
0,2016,15.970978
1,2017,15.506984
2,2018,13.633929
3,2019,10.685489
4,2020,11.416173
5,2021,13.482787
6,2022,10.835999
7,2023,9.299065
8,2024,8.945289
9,2025,8.514464


### Key Findings — Análise 15

**1. Vida tem o menor LR entre os 3 produtos em todas as regiões, sem exceção**
Mesmo a região "pior" para Vida (Sul, 10,1%) fica muito abaixo do LR médio nacional de Auto (65,2%). O produto é estruturalmente de baixo risco em qualquer geografia.

**2. Bahia e Pernambuco combinam o menor LR com o maior crescimento**
BA (LR 3,2%) e PE (LR 4,0%) são os dois estados de menor sinistralidade do Top 10 — e, pela Análise 14, também os de maior crescimento (+18,8% e +29,1% YoY). É a combinação ideal de expansão geográfica com baixo risco.

**3. MG e RJ concentram o maior LR do produto, mas em patamares ainda confortáveis**
Os "piores" estados (MG 14,0%, RJ 13,2%) seguem muito abaixo da média de qualquer região de Auto ou Residencial — não há um ponto de alerta regional real em Vida.

**4. Tendência de queda estrutural do LR: de 16,0% (2016) para 8,5% (2025)**
É a trajetória de melhora mais consistente entre os 3 produtos — sem os saltos de reversão observados em Auto (pandemia + rebote 2021) ou a oscilação lateral do Residencial.

> **Gráfico ThinkCell sugerido:** Barra horizontal (LR por região/UF) + linha de tendência do LR anual, destacando a trajetória de queda contínua.

## Análise 16 — Vida: Ranking de Competidores no Produto (Rolling 12M: mai/2025–abr/2026)

> **Subtítulo do slide:** Bradesco Vida e Prudential formam um duopólio que controla 65% do mercado de Vida

Top 10 seguradoras por prêmio direto de Vida, com classificação **especialista vs. generalista** e a posição isolada da Youse neste produto.

In [25]:
a16_cia = (
    dfp_rolling
    .groupby('coenti', as_index=False)
    .agg(premio_dir=('premio_dir','sum'), sin_dir=('sin_dir','sum'))
)
a16_cia = a16_cia.merge(ses_cias[['Coenti','Noenti']], left_on='coenti', right_on='Coenti', how='left')
a16_cia['noenti']         = a16_cia['Noenti'].fillna('Empresa ' + a16_cia['coenti'].astype(str))
a16_cia['loss_ratio_pct'] = a16_cia['sin_dir'] / a16_cia['premio_dir'] * 100
a16_cia['premio_B']       = a16_cia['premio_dir'] / 1e9
a16_cia = a16_cia.sort_values('premio_dir', ascending=False).reset_index(drop=True)

tot_mercado_produto = a16_cia['premio_dir'].sum()
n_empresas_produto  = len(a16_cia)

# Merge com o consolidado (3 produtos, Análise 4) para classificar especialista vs. generalista
a16_export = a16_cia[['coenti','noenti','premio_dir','premio_B','loss_ratio_pct']].head(10).copy()
a16_export = a16_export.merge(
    a4_cia[['coenti','premio_dir']].rename(columns={'premio_dir':'premio_total_3produtos'}),
    on='coenti', how='left'
)
a16_export['premio_total_3produtos'] = a16_export['premio_total_3produtos'].fillna(a16_export['premio_dir'])
a16_export['pct_portfolio']  = a16_export['premio_dir'] / a16_export['premio_total_3produtos'] * 100
a16_export['perfil']         = np.where(a16_export['pct_portfolio'] >= 70, 'Especialista', 'Generalista')
a16_export['share_pct']      = a16_export['premio_B'] / (tot_mercado_produto / 1e9) * 100
a16_export['share_acum']     = a16_export['share_pct'].cumsum()
top10_share_produto = a16_export['share_pct'].sum()

print(f"{produto_sel} — Ranking de Competidores (mai/2025–abr/2026)")
print(f"Mercado do produto: R$ {tot_mercado_produto/1e9:.2f}B | {n_empresas_produto} empresas ativas")
print(f"\n{'#':<3}{'Empresa':<40}{'Prêmio':>9}{'Share':>7}{'Acum.':>7}{'LR':>7}  Perfil (% portfólio)")
print("-" * 95)
for i, (_, r) in enumerate(a16_export.iterrows(), 1):
    nome = r['noenti'][:39]
    print(f"{i:<3}{nome:<40}{r['premio_B']:>7.2f}B{r['share_pct']:>6.1f}%{r['share_acum']:>6.1f}%"
          f"{r['loss_ratio_pct']:>6.1f}%  {r['perfil']} ({r['pct_portfolio']:.0f}%)")

print(f"\n  >> Top 10 concentra {top10_share_produto:.1f}% do mercado de {produto_sel}")

display(
    a16_export[['noenti','premio_B','share_pct','share_acum','loss_ratio_pct','pct_portfolio','perfil']]
    .rename(columns={
        'noenti':'Empresa','premio_B':'Premio_B','share_pct':'Share_pct',
        'share_acum':'Share_Acumulado','loss_ratio_pct':'LossRatio_pct',
        'pct_portfolio':'Pct_Portfolio_no_Produto','perfil':'Perfil'
    })
    .reset_index(drop=True)
)

# Posição isolada da Youse no produto
youse_prem_p = dfp_rolling[dfp_rolling['coenti'] == YOUSE_COENTI]['premio_dir'].sum()
youse_sin_p  = dfp_rolling[dfp_rolling['coenti'] == YOUSE_COENTI]['sin_dir'].sum()
youse_B_p    = youse_prem_p / 1e9
youse_shr_p  = youse_B_p / (tot_mercado_produto / 1e9) * 100
youse_rank_idx_p = a16_cia[a16_cia['coenti'] == YOUSE_COENTI].index
youse_rank_p = int(youse_rank_idx_p[0]) + 1 if len(youse_rank_idx_p) > 0 else None

print(f"\n{'='*60}")
print(f"Posição da Youse em {produto_sel} (mai/2025–abr/2026)")
print(f"{'='*60}")
if youse_prem_p > 0:
    youse_lr_p = youse_sin_p / youse_prem_p * 100
    rank_str = f"#{youse_rank_p}" if youse_rank_p else "n/d"
    print(f"Prêmio: R$ {youse_B_p:.3f}B | Share: {youse_shr_p:.2f}% | LR: {youse_lr_p:.1f}% | Rank: {rank_str} de {n_empresas_produto}")
else:
    print(f"Youse (coenti {YOUSE_COENTI}) não registra prêmio direto individualizado em {produto_sel} nesta base.")
    print("Consistente com a Análise 4: as operações da Youse são reportadas dentro do grupo Caixa Seguridade na SUSEP.")

Vida — Ranking de Competidores (mai/2025–abr/2026)
Mercado do produto: R$ 25.19B | 76 empresas ativas

#  Empresa                                    Prêmio  Share  Acum.     LR  Perfil (% portfólio)
-----------------------------------------------------------------------------------------------
1  BRADESCO VIDA E PREVIDÊNCIA S.A.           9.46B  37.5%  37.5%   3.1%  Especialista (100%)
2  PRUDENTIAL DO BRASIL SEGUROS S.A.          6.85B  27.2%  64.7%  10.1%  Especialista (100%)
3  MONGERAL AEGON SEGUROS E PREVIDÊNCIA S.    1.57B   6.2%  71.0%   5.3%  Especialista (100%)
4  METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA    1.54B   6.1%  77.1%   4.4%  Especialista (100%)
5  BRASILSEG COMPANHIA DE SEGUROS             1.14B   4.5%  81.6%  10.9%  Especialista (99%)
6  PORTO SEGURO COMPANHIA DE SEGUROS GERAI    0.78B   3.1%  84.7%  37.6%  Generalista (4%)
7  ICATU SEGUROS S.A                          0.63B   2.5%  87.2%   5.9%  Especialista (100%)
8  SICOOB SEGURADORA DE VIDA E PREVIDÊNCIA    0.51B

,Empresa,Premio_B,Share_pct,Share_Acumulado,LossRatio_pct,Pct_Portfolio_no_Produto,Perfil
0,BRADESCO VIDA E PREVIDÊNCIA S.A.,9.458735,37.542731,37.542731,3.128101,100.000000,Especialista
1,PRUDENTIAL DO BRASIL SEGUROS S.A.,6.845606,27.170946,64.713677,10.093449,100.000000,Especialista
2,MONGERAL AEGON SEGUROS E PREVIDÊNCIA S. A.,1.571385,6.236996,70.950673,5.339991,100.000000,Especialista
3,METROPOLITAN LIFE SEGUROS E PREVIDÊNCIA,1.540556,6.114630,77.065303,4.386757,100.000000,Especialista
4,BRASILSEG COMPANHIA DE SEGUROS,1.144887,4.544178,81.609481,10.948565,99.418720,Especialista
5,PORTO SEGURO COMPANHIA DE SEGUROS GERAIS,0.779097,3.092318,84.701799,37.576446,4.196116,Generalista
6,ICATU SEGUROS S.A,0.631375,2.505996,87.207795,5.899036,100.000000,Especialista
7,SICOOB SEGURADORA DE VIDA E PREVIDÊNCIA S.A.,0.506666,2.011012,89.218807,28.718288,100.000000,Especialista
8,ITAU SEGUROS S.A.,0.331435,1.315501,90.534308,14.328442,100.000000,Especialista
9,CAIXA VIDA E PREVIDÊNCIA S.A.,0.314826,1.249576,91.783884,7.292406,100.000000,Especialista



Posição da Youse em Vida (mai/2025–abr/2026)
Youse (coenti 1121) não registra prêmio direto individualizado em Vida nesta base.
Consistente com a Análise 4: as operações da Youse são reportadas dentro do grupo Caixa Seguridade na SUSEP.


### Key Findings — Análise 16

**1. Vida é dominado por um duopólio: Bradesco Vida + Prudential somam 64,7% do mercado**
Bradesco Vida e Previdência (37,5%) e Prudential do Brasil (27,2%) concentram, sozinhas, quase dois terços do prêmio de Vida — uma concentração de mercado muito mais forte que em Auto (líder com 26,0%) ou Residencial (líder com 19,1%).

**2. 9 dos 10 maiores concorrentes são especialistas puros em Vida**
Diferente de Residencial (mix equilibrado) e principalmente de Auto (onde até os "generalistas" têm >85% do portfólio no produto), em Vida quase todo o Top 10 tem 99-100% do próprio portfólio consolidado concentrado neste produto — é o ecossistema competitivo mais "puro" dos três.

**3. Porto Seguro é a exceção — e tem, de longe, o pior LR do Top 10**
A única generalista do ranking (apenas 4% do seu portfólio é Vida) tem LR de 37,6%, quatro a doze vezes pior que os especialistas (3,1%-14,3%). Reforça que a vantagem competitiva da Porto Seguro está concentrada em Auto, não em Vida.

**4. Bradesco Vida combina escala e eficiência extrema: líder de mercado com LR de apenas 3,1%**
Não é um trade-off entre tamanho e sinistralidade — a maior seguradora do produto também é uma das mais eficientes tecnicamente, uma referência de benchmarking para a estratégia de subscrição da Youse.

> **Nota de dados:** Youse (coenti 1121) não registra prêmio direto individualizado em Vida nesta base — mesma limitação documentada na Análise 4 (operação reportada dentro do grupo Caixa Seguridade).

> **Gráfico ThinkCell sugerido:** Barra horizontal rankeada por prêmio, com marcador de perfil (especialista/generalista) e linha de share acumulado.